In [ ]:

# import tables_io
import qp
import numpy as np
import h5py
import os
import matplotlib.pyplot as plt
import pandas as pd
import sklearn.metrics
import scipy.stats

# import rail
# from rail.evaluation.dist_to_dist_evaluator import DistToDistEvaluator
# from rail.evaluation.dist_to_point_evaluator import DistToPointEvaluator
# from rail.evaluation.point_to_point_evaluator import PointToPointEvaluator
# from rail.evaluation.single_evaluator import SingleEvaluator
# from rail.core.stage import RailStage
# from rail.core.data import QPHandle, TableHandle, QPOrTableHandle

# DS = RailStage.data_store
# DS.__class__.allow_overwrite = True

In [ ]:
# file = h5py.File(pdfs_file, 'r')
# file.keys

In [ ]:

# ztrue_posts_file = "../paper_data/output_lsstErr_testSet_posts.hdf5"

# ztrue_data = DS.read_file(key='data', handle_class=QPHandle, path=ztrue_posts_file)
# # ztrue_data_2 = DS.read_file('ztrue_data', TableHandle, ztrue_posts_file)

In [ ]:
# point_zs_file = "../paper_data/output_lsst_error (for posts).pq"

# true_point_zs = DS.read_file('true_point_zs', TableHandle, point_zs_file)


# point_zs = pd.read_parquet(point_zs_file)
# point_z_data = point_zs['redshift']

In [ ]:
# import os
# from rail.utils.path_utils import find_rail_file

# test_file = h5py.File(find_rail_file('examples_data/testdata/test_dc2_validation_9816.hdf5'))
# test_path = find_rail_file('examples_data/testdata/test_dc2_validation_9816.hdf5')
# #test_pdfs_file = find_rail_file('examples_data/evaluation_data/data/output_fzboost.hdf5')

# test_file['photometry'].keys()

# test_point_zs = DS.read_file('test_table', TableHandle, test_path)
# # test_ens = DS.read_file(key='pdfs_data', handle_class=QPHandle, path=test_pdfs_file)

# test_point_zs()['photometry']



In [ ]:
import pandas as pd
BOSS = pd.read_parquet("../paper_data/TrainZ/BOSS/output_quantity_cut_1.pq")
print('BOSS: '+str(len(BOSS)))
DEEP2 = pd.read_parquet("../paper_data/TrainZ/DEEP2/output_quantity_cut_1.pq")
print('DEEP2: '+str(len(DEEP2)))
GAMA = pd.read_parquet("../paper_data/TrainZ/GAMA/output_quantity_cut_1.pq")
print('GAMA: '+str(len(GAMA)))
HSC = pd.read_parquet("../paper_data/TrainZ/HSC/output_quantity_cut_1.pq")
print('HSC: '+str(len(HSC)))
VVDSf02 = pd.read_parquet("../paper_data/TrainZ/VVDSf02/output_quantity_cut_1.pq")
print('VVDSf02: '+str(len(VVDSf02)))
zCOSMOS = pd.read_parquet("../paper_data/TrainZ/zCOSMOS/output_quantity_cut_1.pq")
print('zCOSOMS: '+str(len(zCOSMOS)))
LSSTerr = pd.read_parquet("../paper_data/TrainZ/LSSTerr_only/output_quantity_cut_1.pq")
print('LSSTerr only: '+str(len(LSSTerr)))

In [ ]:
import scipy.stats as sps

# evaluate a single distribution's PDF at one value
print("PDF at one point for one distribution:", 
      sps.norm(loc=0, scale=1).pdf(0.5))

# evaluate a single distribution's PDF at multiple value
print("PDF at three points for one distribution:", 
      sps.norm(loc=0, scale=1).pdf([0.5, 1., 1.5]))

# evalute three distributions' PDFs at one shared value
print("PDF at one point for three distributions:", 
      sps.norm(loc=[0., 1., 2.], scale=1).pdf(0.5))

# evalute three distributions' PDFs each at one different value
print("PDF at one different point for three distributions:", 
      sps.norm(loc=[0., 1., 2.], scale=1).pdf([0.5, 1., 1.5]))

# evalute three distributions' PDFs each at four different values
# (note the change in shape of the argument)
print("PDF at four different points for three distributions:\n",
      sps.norm(loc=[0., 1., 2.], scale=1).pdf([[0.5],[1.],[1.5],[2]]))

# evalute three distributions' PDFs at each of four different values
# (note the change in shape of the argument)
print("PDF at four different points for three distributions: broadcast reversed\n",
      sps.norm(loc=[[0.], [1.], [2.]], scale=1).pdf([0.5,1.,1.5,2]))

# Plots

## different degraders, same estimators

In [ ]:
spec_ls = ['BOSS', 'DEEP2', 'GAMA', 'HSC', 'VVDSf02', 'zCOSMOS']
pivot_ls = ["1.0", "1.4", 'control'] 
est_ls = ['TrainZ', 'CMNN', 'GPz', 'PZFlow', 'FZBoost']

def makeSpecFiles(est):
    file_ls = []
    for i in spec_ls:
        file = h5py.File('../paper_data/specSelection_lsstErr_'+est+'/outputs/'+i+'/output_estimate_'+est+'.hdf5', 'r')
        file_ls.append(file)
    return file_ls

def makeInvzFiles(est):
    file_ls = []
    for i in pivot_ls:
        file = h5py.File('../paper_data/invz_lsstErr_'+est+'/outputs/'+i+'/output_estimate_'+est+'.hdf5', 'r')
        file_ls.append(file)
    return file_ls

def makePlots(est, type, dim, size):
    if type == 'invz':
        makeFiles = makeInvzFiles
        name_ls = pivot_ls
    elif type == 'spec':
        makeFiles = makeSpecFiles
        name_ls = spec_ls

    control_file = makeInvzFiles(est)[2]

    fig, axes = plt.subplots(nrows = dim, ncols = dim, figsize = (size, (2/3)*size)) 
    ct = 0
    for i in range(0, dim):
        for j in range(0, dim):
            axes[i][j].plot(ztrue_posts['meta']['xvals'][0], ztrue_posts['data']['yvals'][ct], label = "test set pdf") 
            axes[i][j].plot(point_zs['redshift'][ct]*np.ones(101), np.linspace(0, 30, 101), color = 'k', label = 'true redshift')

            #max = np.max(ztrue_posts['data']['yvals'][ct])
            #ind = np.where(ztrue_posts['data']['yvals'] == max)
            # axes[i][j].plot( ztrue_posts['meta']['xvals'][0][ind[1][0]] * np.ones(len(ztrue_posts['meta']['xvals'][0])), np.linspace(0, 20, 101) , color = 'k')

            if est == 'CMNN' or est == 'GPz':
                arr = ztrue_posts['meta']['xvals'][0]
                val = 0
                for file in makeFiles(est):
                    axes[i][j].plot(arr, (1. /(file['data']['scale'][ct] * np.sqrt(2*np.pi))) * np.exp((-1/2) * ((arr - file['data']['loc'][ct])/ file['data']['scale'][ct] )**2) , label = name_ls[val])
                    val += 1
                    axes[i][j].plot(file['ancil']['zmode'][ct]*np.ones(101), np.linspace(0, 30, 101), linestyle = 'dashed', color = 'gray')
                if type != 'invz':
                    axes[i][j].plot(arr, (1. /(control_file['data']['scale'][ct] * np.sqrt(2*np.pi))) * np.exp((-1/2) * ((arr - control_file['data']['loc'][ct])/ control_file['data']['scale'][ct] )**2) , label = 'control')
                axes[0][0].legend()

            if est == 'TrainZ' or est == 'FZBoost':
                val = 0
                for file in makeFiles(est):
                    axes[i][j].plot(file['meta']['xvals'][0], file['data']['yvals'][ct], label = name_ls[val])
                    val += 1
                    axes[i][j].plot(file['ancil']['zmode'][ct]*np.ones(101), np.linspace(0, 30, 101), linestyle = 'dashed', color = 'gray')
                if type != 'invz':
                    axes[i][j].plot(control_file['meta']['xvals'][0], control_file['data']['yvals'][ct], label = 'control')
                axes[0][0].legend()

            ct += 1
    if type == 'spec':
        fig.suptitle(est+" Estmiates for Spectrascopic Sky Surveys", size = 'xx-large')
    elif type == 'invz':
        fig.suptitle(est+" Estmiates for Inverse Redshift Incompleteness", size = 'xx-large')
    fig.supxlabel("Redshift", size = 'x-large')
    #fig.supylabel("pdf", size = 'x-large')
    axes[0][0].set_ylabel("pdf")

In [ ]:
makePlots('TrainZ', 'spec', 4, 24)

In [ ]:
makePlots('CMNN', 'spec', 4, 24)

In [ ]:
makePlots('GPz','spec', 4, 24)

# file = h5py.File('../paper_data/specSelection_lsstErr_FZBoost/outputs/BOSS/output_estimate_FZBoost.hdf5', 'r')

# file['data']['yvals']

In [ ]:
makePlots('FZBoost','spec', 4, 24)

In [ ]:
makePlots('TrainZ','invz', 4, 24)

In [ ]:
makePlots('CMNN','invz', 4, 24)

In [ ]:
makePlots('GPz','invz', 5, 24)

In [ ]:
makePlots('FZBoost','invz', 5, 24)

## different estimators, same degraders

In [ ]:
est_ls = ['TrainZ', 'CMNN', 'GPz', "FZBoost"]
spec_ls = ['BOSS', 'DEEP2', 'GAMA', 'HSC', 'VVDSf02', 'zCOSMOS']
pivot_ls = ['1.0', '1.4', 'control']

def makeFiles(survey_pivot, type):
    spec_files = []
    invz_files = []
    if type == 'spec':
        for est in est_ls:
            spec_file = h5py.File('../paper_data/specSelection_lsstErr_'+est+'/outputs/'+survey_pivot+'/output_estimate_'+est+'.hdf5', 'r')
            spec_files.append(spec_file)
        return spec_files
    if type == 'invz':
        for est in est_ls:
            invz_file = h5py.File('../paper_data/invz_lsstErr_'+est+'/outputs/'+survey_pivot+'/output_estimate_'+est+'.hdf5', 'r')
            invz_files.append(invz_file)
        return invz_files

def makePlots2(survey_pivot, type, dim, size):

    fig, axes = plt.subplots(nrows = dim, ncols = dim, figsize = (size, (2/3)*size)) 
    ct = 0
    for i in range(0, dim):
        for j in range(0, dim):
            axes[i][j].plot(ztrue_posts['meta']['xvals'][0], ztrue_posts['data']['yvals'][ct], label = "test set pdf") 
            axes[i][j].plot(true_point_zs['redshift'][ct]*np.ones(101), np.linspace(0, 30, 101), color = 'k', label = 'true redshift')

            val = 0
            for file in makeFiles(survey_pivot, type):
                #print(file)
                if 'TrainZ' in str(file) or 'FZBoost' in str(file):
                    axes[i][j].plot(file['meta']['xvals'][0], file['data']['yvals'][ct], label = est_ls[val])
                if 'CMNN' in str(file) or 'GPz' in str(file):
                    arr = ztrue_posts['meta']['xvals'][0]
                    axes[i][j].plot(arr, (1. /(file['data']['scale'][ct] * np.sqrt(2*np.pi))) * np.exp((-1/2) * ((arr - file['data']['loc'][ct])/ file['data']['scale'][ct] )**2) , label = est_ls[val])
                val += 1
            axes[0][0].legend()
            ct += 1
    if type == 'spec':
        fig.suptitle(survey_pivot+" Training Set Degradation Estimations", size = 'xx-large')
    elif type == 'invz' and survey_pivot == 'control':
        fig.suptitle("Undegraded Training Set Estimations", size = 'xx-large')
    else:
        fig.suptitle("Inverse Redshift Incompleteness, pivot = "+survey_pivot+" Estimations", size = 'xx-large')
    fig.supxlabel("Redshift", size = 'x-large')
    #fig.supylabel("pdf", size = 'x-large')
    axes[0][0].set_ylabel("pdf")

In [ ]:
makePlots2('BOSS', 'spec', 5, 24)

In [ ]:
makePlots2('DEEP2', 'spec', 5, 24)

In [ ]:
makePlots2('GAMA', 'spec', 5, 24)

In [ ]:
makePlots2('HSC', 'spec', 5, 24)

In [ ]:
makePlots2('VVDSf02', 'spec', 5, 24)

In [ ]:
makePlots2('control', 'invz', 7, 24)

In [ ]:
makePlots2('zCOSMOS', 'spec', 5, 24)

In [ ]:
makePlots2('1.0', 'invz', 5, 24)

In [ ]:
makePlots2('1.4', 'invz', 5, 24)

In [ ]:
makePlots2('control', 'invz', 5, 24)

# Dist-to-dist

In [ ]:
import scipy.stats 
import sklearn.metrics
import scipy.special

In [ ]:
deg_ls = ['BOSS', 
          'DEEP2', 
          'GAMA', 
          'HSC', 
          'VVDSf02', 
          'zCOSMOS', 
          'invz=0.1', 
          'invz=0.3',
          'invz=0.6',
          'invz=1.0', 
          'invz=1.4', 
          'LSSTerr_only', 
          ]

In [ ]:
def writeToFiles(est, cvm_dict, ks_dict, rmse_dict, kld_dict):
    if est == 'GPz':
        os.chdir('../paper_data/GPz_GL')
    else: 
        os.chdir('../paper_data/'+est)
        
    cvm_df = pd.DataFrame.from_dict(cvm_dict, orient = 'columns')
    if est == 'GPz':
        cvm_path = '../paper_data/GPz_GL/cvm_stats.pq'
    else:
        cvm_path = '../paper_data/'+est+'/cvm_stats.pq'
    cvm_df.to_parquet(cvm_path)

    ks_df = pd.DataFrame.from_dict(ks_dict, orient = 'columns')
    if est =='GPz': 
        ks_path = '../paper_data/GPz_GL/ks_stats.pq'
    else:
        ks_path = '../paper_data/'+est+'/ks_stats.pq'
    ks_df.to_parquet(ks_path)

    rmse_df = pd.DataFrame.from_dict(rmse_dict, orient = 'columns')
    if est == 'GPz':
        rmse_path = '../paper_data/GPz_GL/rmse_stats.pq'
    else:
        
        rmse_path = '../paper_data/'+est+'/rmse_stats.pq'
    rmse_df.to_parquet(rmse_path)

    kld_df = pd.DataFrame.from_dict(kld_dict, orient = 'columns')
    if est == 'GPz':
        kld_path = '../paper_data/GPz_GL/kld_stats.pq'
    else:
        kld_path = '../paper_data/'+est+'/kld_stats.pq'
    kld_df.to_parquet(kld_path)


In [ ]:
def writeToFiles_tq(est, cvm_dict, ks_dict, rmse_dict, kld_dict):

    cvm_df = pd.DataFrame.from_dict(cvm_dict, orient = 'columns')
    if est == 'GPz':
        cvm_path = '../paper_data/GPz_GL/cvm_stats_tq.pq'
    else:
        cvm_path = './'+est+'/cvm_stats_tq.pq'
    cvm_df.to_parquet(cvm_path)

    ks_df = pd.DataFrame.from_dict(ks_dict, orient = 'columns')
    if est =='GPz': 
        ks_path = '../paper_data/GPz_GL/ks_stats_tq.pq'
    else:
        ks_path = './'+est+'/ks_stats_tq.pq'
    ks_df.to_parquet(ks_path)

    rmse_df = pd.DataFrame.from_dict(rmse_dict, orient = 'columns')
    if est == 'GPz':
        rmse_path = '../paper_data/GPz_GL/rmse_stats_tq.pq'
    else:
        
        rmse_path = './'+est+'/rmse_stats_tq.pq'
    rmse_df.to_parquet(rmse_path)

    kld_df = pd.DataFrame.from_dict(kld_dict, orient = 'columns')
    if est == 'GPz':
        kld_path = '../paper_data/GPz_GL/kld_stats_tq.pq'
    else:
        kld_path = './'+est+'/kld_stats_tq.pq'
    kld_df.to_parquet(kld_path)


In [ ]:
def removeNansInfs(df, cols):
    ls = []
    for col in cols: 
        ls +=  df.index[np.isnan(df[col]) ==True].tolist() + df.index[np.isfinite(df[col]) ==False].tolist() 

    drops = list(set(ls))
    new_df = df.drop(drops)

    return new_df 


## TrainZ

In [ ]:
TrainZ_cvm_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             'control': []
             }

In [ ]:
TrainZ_ks_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             'control': []
             }

In [ ]:
TrainZ_rmse_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             'control': []
             }

In [ ]:
TrainZ_kld_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             'control': []
             }

In [ ]:
TrainZ_fails_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             'control': []
             }

In [ ]:
T_ks = {'invz=0.3': []} 
T_cvm = {'invz=0.3': []}
T_rmse = {'invz=0.3': []}
T_kld = {'invz=0.3': []} 

## Wasserstein Distance

In [ ]:
TrainZ_wdist_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1':[],
             'invz=0.3':[],
             'invz=0.6':[],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             }

In [ ]:
CMNN_wdist_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1':[],
             'invz=0.3':[],
             'invz=0.6':[],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             }

In [ ]:
GPz_wdist_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1':[],
             'invz=0.3':[],
             'invz=0.6':[],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             }

In [ ]:
FZBoost_wdist_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1':[],
             'invz=0.3':[],
             'invz=0.6':[],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             }

In [ ]:
PzFlow_wdist_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1':[],
             'invz=0.3':[],
             'invz=0.6':[],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             }

In [ ]:
!pwd

In [ ]:
os.getcwd()

In [ ]:
import scipy.stats as sts
import pandas as pd
import numpy as np
import os
import qp
import h5py

def getTrainZWDist(deg):
    true_posts_file = "../paper_data/output_lsstErr_test_data_posts.hdf5"
    gals = pd.read_parquet("./output_quantity_cut_1.pq")['redshift'].index.tolist()

#     os.chdir("../paper_data/TrainZ/"+deg)
    pdfs_file = "../paper_data/TrainZ/"+deg+"/output_estimate_TrainZ.hdf5"

    truth_ens = qp.read(true_posts_file)
    est_ens = qp.read(pdfs_file)
    
    xvals = h5py.File(true_posts_file)['meta']['xvals'][0]

    wdist_ls = []
    fails = []

    for i in range(0, len(gals)-1):

        # the true posterior is the array-like first argument
        truth = h5py.File(true_posts_file)['data']['yvals'][i].tolist()

        yvals = h5py.File(pdfs_file)['data']['yvals'][i].tolist()

        if np.isnan(yvals[0])==True:
            fails.append(gals[i])
            print(deg, gals[i]) 
            wdist_ls.append(np.nan)

        elif np.isnan(truth[0])==True:
            wdist_ls.append(np.nan)
            pass

        else:   
#             wdist = sts.wasserstein_distance(truth_ens[i].cdf(xvals)[0], est_ens[i].cdf(xvals)[0])
#             print(truth_ens[i].rvs(size = 1000))
            wdist = sts.wasserstein_distance(truth_ens[i].rvs(size = 1000)[0], est_ens[i].rvs(size = 1000)[0])
            wdist_ls.append(wdist)
    
    TrainZ_wdist_dict[deg] = wdist_ls

    print(deg +' done')


In [ ]:
for deg in deg_ls:
    getTrainZWDist(deg)

In [ ]:
T_wdist_df = pd.DataFrame.from_dict(TrainZ_wdist_dict)
T_wdist_df.to_parquet('../paper_data/TrainZ/wdist_stats_tq.pq')

In [ ]:
def getCMNNWDist(deg):
    true_posts_file = "../paper_data/output_lsstErr_test_data_posts.hdf5"
    gals = pd.read_parquet("output_quantity_cut_1.pq")['redshift'].index.tolist()

#     os.chdir("../paper_data/CMNN/"+deg)
    pdfs_file = "../paper_data/CMNN/"+deg+"/output_estimate_CMNN.hdf5"

    truth_ens = qp.read(true_posts_file)
    est_ens = qp.read(pdfs_file)
    
    xvals = h5py.File(true_posts_file)['meta']['xvals'][0]

    wdist_ls = []
    fails = []

    for i in range(0, len(gals)-1):

        # the true posterior is the array-like first argument
        truth = h5py.File(true_posts_file)['data']['yvals'][i].tolist()

#         yvals = h5py.File(pdfs_file)['data']['yvals'][i].tolist()
#         print(yvals)

#         if np.isnan(yvals[0])==True:
#             fails.append(gals[i])
#             print(deg, gals[i]) 
#             wdist_ls.append(np.nan)

#         elif np.isnan(truth[0])==True:
#             wdist_ls.append(np.nan)
#             pass

#         else:   
        wdist = sts.wasserstein_distance(truth_ens[i].rvs(size = 1000)[0], est_ens[i].rvs(size = 1000)[0])
        wdist_ls.append(wdist)
    
    CMNN_wdist_dict[deg] = wdist_ls

    print(deg +' done')



In [ ]:

for deg in deg_ls:
    getCMNNWDist(deg)


In [ ]:
C_wdist_df = pd.DataFrame.from_dict(CMNN_wdist_dict)
C_wdist_df.to_parquet('../paper_data/CMNN/wdist_stats_tq.pq')


In [ ]:
def getGPzWDist(deg):
    true_posts_file = "../paper_data/output_lsstErr_test_data_posts.hdf5"
    gals = pd.read_parquet("output_quantity_cut_1.pq")['redshift'].index.tolist()

#     os.chdir("../paper_data/GPz/"+deg)
    pdfs_file = "../paper_data/GPz_GL/"+deg+"/output_estimate_GPz.hdf5"

    truth_ens = qp.read(true_posts_file)
    est_ens = qp.read(pdfs_file)
    
    xvals = h5py.File(true_posts_file)['meta']['xvals'][0]

    wdist_ls = []
    fails = []

    for i in range(0, len(gals)-1):

        # the true posterior is the array-like first argument
        truth = h5py.File(true_posts_file)['data']['yvals'][i].tolist()

        #yvals = h5py.File(pdfs_file)['data']['yvals'][i].tolist()

        # if np.isnan(yvals[0])==True:
        #     fails.append(gals[i])
        #     print(deg, gals[i]) 
        #     wdist_ls.append(np.nan)

        if np.isnan(truth[0])==True:
            wdist_ls.append(np.nan)
            pass

        else:   
            wdist = sts.wasserstein_distance(truth_ens[i].rvs(size = 1000)[0], est_ens[i].rvs(size = 1000)[0])
            wdist_ls.append(wdist)
    
    GPz_wdist_dict[deg] = wdist_ls

    print(deg +' done')



for deg in deg_ls:
    getGPzWDist(deg)

In [ ]:
G_wdist_df = pd.DataFrame.from_dict(GPz_wdist_dict)
G_wdist_df.to_parquet('../paper_data/GPz_GL/wdist_stats_tq.pq')
G_wdist_df

In [ ]:
def getFZBoostWDist(deg):
    true_posts_file = "../paper_data/output_lsstErr_test_data_posts.hdf5"
    gals = pd.read_parquet("./output_quantity_cut_1.pq")['redshift'].index.tolist()

#     os.chdir("../paper_data/FZBoost/"+deg)
    pdfs_file = "../paper_data/FZBoost/"+deg+"/output_estimate_FZBoost.hdf5"

    truth_ens = qp.read(true_posts_file)
    est_ens = qp.read(pdfs_file)
    
    xvals = h5py.File(true_posts_file)['meta']['xvals'][0]

    wdist_ls = []
    fails = []

    for i in range(0, len(gals)-1):

        # the true posterior is the array-like first argument
        truth = h5py.File(true_posts_file)['data']['yvals'][i].tolist()

        yvals = h5py.File(pdfs_file)['data']['yvals'][i].tolist()

#         if np.isnan(yvals[0])==True:
#             fails.append(gals[i])
#             print(deg, gals[i]) 
#             wdist_ls.append(np.nan)

#         if np.isnan(truth[0])==True:
#             wdist_ls.append(np.nan)
#             pass

#         else:   
        wdist = sts.wasserstein_distance(truth_ens[i].rvs(size = 1000)[0], est_ens[i].rvs(size = 1000)[0])
        wdist_ls.append(wdist)
    
    FZBoost_wdist_dict[deg] = wdist_ls

    print(deg +' done')



for deg in deg_ls:
    getFZBoostWDist(deg)

In [ ]:
F_wdist_df = pd.DataFrame.from_dict(FZBoost_wdist_dict)
F_wdist_df.to_parquet('../paper_data/FZBoost/wdist_stats_tq.pq')
F_wdist_df

In [ ]:
def getPzFlowWDist(deg):
    true_posts_file = "../paper_data/output_lsstErr_test_data_posts.hdf5"
    gals = pd.read_parquet("./output_quantity_cut_1.pq")['redshift'].index.tolist()

#     os.chdir("../paper_data/PzFlow/"+deg)
    pdfs_file = "../paper_data/PZFlow/"+deg+"/output_estimate_pzflow.hdf5"

    truth_ens = qp.read(true_posts_file)
    est_ens = qp.read(pdfs_file)
    
    xvals = h5py.File(true_posts_file)['meta']['xvals'][0]

    wdist_ls = []
    fails = []

    for i in range(0, len(gals)-1):

        # the true posterior is the array-like first argument
        truth = h5py.File(true_posts_file)['data']['yvals'][i].tolist()

        yvals = h5py.File(pdfs_file)['data']['yvals'][i].tolist()

        if np.isnan(yvals[0])==True:
            fails.append(gals[i])
            print(deg, gals[i]) 
            wdist_ls.append(np.nan)

        elif np.isnan(truth[0])==True:
            wdist_ls.append(np.nan)
            pass

        else:   
            wdist = sts.wasserstein_distance(truth_ens[i].rvs(size = 1000)[0], est_ens[i].rvs(size = 1000)[0])
            wdist_ls.append(wdist)
    
    PzFlow_wdist_dict[deg] = wdist_ls

    print(deg +' done')



for deg in deg_ls:
    getPzFlowWDist(deg)

In [ ]:
P_wdist_df = pd.DataFrame.from_dict(PzFlow_wdist_dict)
P_wdist_df.to_parquet('../paper_data/PZFlow/wdist_stats_tq.pq')
P_wdist_df

## TrainZ CvM, KS, RMSE, KLD

In [ ]:
def getTrainZDistStats(deg):

    true_posts_file = "../paper_data/output_lsstErr_test_data_posts.hdf5"
    gals = pd.read_parquet("./output_quantity_cut_1.pq")['redshift'].index.tolist()

#     os.chdir("../paper_data/TrainZ/"+deg)
    pdfs_file = "../paper_data/TrainZ/"+deg+"/output_estimate_TrainZ.hdf5"

    truth_ens = qp.read(true_posts_file)
    est_ens = qp.read(pdfs_file)

    xvals = h5py.File(true_posts_file)['meta']['xvals'][0]
    
    cvm_ls = []
    ks_ls = []
    rmse_ls = []
    kld_ls = []

    fails = []
    print(f'start {deg}')
    
    for i in range(0, len(gals)-1):
        
        if i%1000==0:
            print(f'processing gal {i}')
        
        # the true posterior is the array-like first argument
        truth = h5py.File(true_posts_file)['data']['yvals'][i].tolist()

        yvals = h5py.File(pdfs_file)['data']['yvals'][i].tolist()

        if np.isnan(yvals[0])==True:
            fails.append(gals[i])
            print(deg, gals[i]) 

            cvm_ls.append(np.nan)
            ks_ls.append(np.nan)
            rmse_ls.append(np.nan)
            kld_ls.append(np.nan)

        elif np.isnan(truth[0])==True:
            cvm_ls.append(np.nan)
            ks_ls.append(np.nan)
            rmse_ls.append(np.nan)
            kld_ls.append(np.nan)
            pass

        else:      

            cvm = scipy.stats.cramervonmises(truth_ens[i].rvs(size = 1000)[0], est_ens[i].cdf)
            cvm_ls.append(cvm.statistic)
#             print('cvm for '+spec+' done')

            ks = scipy.stats.kstest(truth_ens[i].rvs(size = 1000)[0], est_ens[i].cdf(xvals)[0])
            ks_ls.append(ks.statistic)
#             print('ks for '+spec+' done')

            rmse = sklearn.metrics.mean_squared_error(truth_ens[i].pdf(xvals)[0], est_ens[i].pdf(xvals)[0])
            rmse_ls.append(rmse)
#             print('rmse for '+spec+' done')

            truth_data = truth_ens[i].pdf(xvals)[0]
            est_data = est_ens[i].pdf(xvals)[0] 

            truth_data[truth_data < 1e-300] = 1e-300
            est_data[est_data < 1e-300] = 1e-300

            kld = scipy.special.kl_div(truth_data, est_data)
            kld_ls.append(np.sum(kld) * 3.0 / len(gals))
#             print('kld for '+spec+' done')

        
    TrainZ_cvm_dict[deg] = cvm_ls
    TrainZ_ks_dict[deg] = ks_ls
    TrainZ_rmse_dict[deg] = rmse_ls
    TrainZ_kld_dict[deg] = kld_ls
    TrainZ_fails_dict[deg] = fails
#     T_ks[deg] = ks_ls
#     T_cvm[deg] = cvm_ls
#     T_rmse[deg] = rmse_ls
#     T_kld[deg] = kld_ls

    print(deg+' done')


In [ ]:
extra_invz = deg_ls + ['LSSTerr_null']
# extra_invz = ['LSSTerr_null']

In [ ]:
for deg in extra_invz:
    getTrainZDistStats(deg)

In [ ]:
for key in TrainZ_cvm_dict.keys():
    print(key, len(TrainZ_cvm_dict[key]))

In [ ]:
len(TrainZ_cvm_dict['BOSS']), len(TrainZ_cvm_dict['BOSS'])

In [ ]:
# for dict_ in [ TrainZ_cvm_dict, TrainZ_ks_dict, TrainZ_rmse_dict, TrainZ_kld_dict]:
#     dict_.pop('control') 

writeToFiles_tq('TrainZ',  TrainZ_cvm_dict, TrainZ_ks_dict, TrainZ_rmse_dict, TrainZ_kld_dict)


In [ ]:
P = [.05, .1, .2, .05, .15, .25, .08, .12]
Q = [.3, .1, .2, .1, .1, .02, .08, .1]

print(sum(scipy.special.kl_div(P, Q)))
print(sum(scipy.special.rel_entr(P, Q)))

In [ ]:
# pd.read_parquet('../paper_data/TrainZ/kld_stats.pq')

In [ ]:
T_df_tmp = pd.read_parquet('../paper_data/TrainZ/cvm_stats.pq')
T_df = T_df_tmp#.drop(columns='control') 
T_df.insert(8, 'invz=0.3', T_cvm['invz=0.3'])
T_df.to_parquet('../paper_data/TrainZ/cvm_stats.pq')
T_df

In [ ]:
for deg in deg_ls: 
    getTrainZDistStats(deg)

In [ ]:
writeToFiles('TrainZ', TrainZ_kld_dict) #TrainZ_cvm_dict, TrainZ_ks_dict, TrainZ_rmse_dict, TrainZ_kld_dict)

In [ ]:
df = pd.read_parquet('../paper_data/FZBoost/kld_stats.pq')

for i in df['BOSS']:
    if np.isfinite(i)==False and np.isnan(i)==False:
        print(i)

## CMNN

In [ ]:
CMNN_cvm_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             }

In [ ]:
CMNN_ks_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             }

In [ ]:
CMNN_rmse_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             }

In [ ]:
CMNN_kld_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             }

In [ ]:
CMNN_fails_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             }

In [ ]:
C_ks = {'invz=0.3': []}
C_cvm = {'invz=0.3': []}
C_rmse = {'invz=0.3': []}
C_kld = {'invz=0.3': []}

In [ ]:
gals = pd.read_parquet("../paper_data/output_quantity_cut_1.pq")['redshift'].index.tolist()

print(len(gals))

filepath = "../paper_data/FZBoost/BOSS/output_estimate_FZBoost.hdf5"

file = h5py.File(filepath)
print(file['data']['yvals'])

In [ ]:
def getCMNNDistStats(deg):
    print(f'start {deg}')
    true_posts_file = "../paper_data/output_lsstErr_test_data_posts.hdf5"
    gals = pd.read_parquet("./output_quantity_cut_1.pq")['redshift'].index.tolist()

#     os.chdir("../paper_data/CMNN/"+deg)
    pdfs_file = "../paper_data/CMNN/"+deg+"/output_estimate_CMNN.hdf5"

    est_ens = qp.read(pdfs_file)
    truth_ens = qp.read(true_posts_file)

    xvals = h5py.File(true_posts_file)['meta']['xvals'][0]
    
    cvm_ls = []
    ks_ls = []
    rmse_ls = []
    kld_ls = []

    fails = []
    
    for i in range(0, len(gals)-1):
        if i%1000==0:
            print(f'processing gal {i}')

        # the true posterior is the array-like first argument
        truth = h5py.File(true_posts_file)['data']['yvals'][i]

        if np.isnan(h5py.File(pdfs_file)['data']['loc'][i])==True or np.isnan(h5py.File(pdfs_file)['data']['scale'][i])==True:
            fails.append(gals[i])
            print(deg, gals[i])  

            cvm_ls.append(np.nan)
            ks_ls.append(np.nan)
            rmse_ls.append(np.nan)
            kld_ls.append(np.nan)

        elif np.isnan(truth[0])==True:
            cvm_ls.append(np.nan)
            ks_ls.append(np.nan)
            rmse_ls.append(np.nan)
            kld_ls.append(np.nan)
            pass

        else:

            cvm = scipy.stats.cramervonmises(truth_ens[i].rvs(size = 1000)[0], est_ens[i].cdf)
            cvm_ls.append(cvm.statistic)
#             print('cvm for '+spec+' done')

            ks = scipy.stats.kstest(truth_ens[i].rvs(size = 1000)[0], est_ens[i].cdf(xvals)[0])
            ks_ls.append(ks.statistic)
            #print('ks for '+spec+' done')

            rmse = sklearn.metrics.mean_squared_error(truth_ens[i].pdf(xvals)[0], est_ens[i].pdf(xvals)[0])
            rmse_ls.append(rmse)
            #print('rmse for '+spec+' done')

            truth_data = truth_ens[i].pdf(xvals)[0]
            est_data = est_ens[i].pdf(xvals)[0] 

            truth_data[truth_data < 1e-300] = 1e-300
            est_data[est_data < 1e-300] = 1e-300

            kld = scipy.special.kl_div(truth_data, est_data)
            kld_ls.append(np.sum(kld) * 3.0 / len(gals))
            # print('kld for '+spec+' done')

    
    CMNN_cvm_dict[deg] = cvm_ls
    CMNN_ks_dict[deg]=ks_ls
    CMNN_rmse_dict[deg]=rmse_ls
    CMNN_kld_dict[deg] = kld_ls
    #CMNN_fails_dict[deg] = fails
#     C_ks[deg] = ks_ls
#     C_cvm[deg] = cvm_ls
#     C_rmse[deg] = rmse_ls
#     C_kld[deg] = kld_ls
    
    print(deg+" done")


    

In [ ]:
for deg in extra_invz:
    getCMNNDistStats(deg)

In [ ]:
writeToFiles_tq('CMNN',  CMNN_cvm_dict, CMNN_ks_dict, CMNN_rmse_dict, CMNN_kld_dict)




In [ ]:
# C_df_tmp = pd.read_parquet('../paper_data/CMNN/cvm_stats.pq')
# C_df = C_df_tmp#.drop(columns='control') 
# C_df.insert(7, 'invz=0.3', C_cvm['invz=0.3'])
# C_df.to_parquet('../paper_data/CMNN/cvm_stats.pq')
# C_df

In [ ]:
for deg in deg_ls: 
    getCMNNDistStats(deg)

In [ ]:
writeToFiles('CMNN', CMNN_kld_dict) #CMNN_cvm_dict, CMNN_ks_dict, CMNN_rmse_dict, CMNN_kld_dict)

In [ ]:
pd.read_parquet('../paper_data/CMNN/kld_stats.pq')

## GPz

### GL

In [ ]:
GPz_GL_cvm_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [],
             'invz=0.3': [],
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             }
             
GPz_GL_ks_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [],
             'invz=0.3': [],
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             }
             
GPz_GL_rmse_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [],
             'invz=0.3': [],
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             }
             
GPz_GL_kld_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [],
             'invz=0.3': [],
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             }
             

In [ ]:
def getGPzGLDistStats(deg):
    print(deg)
    true_posts_file = "../paper_data/output_lsstErr_test_data_posts.hdf5"
    gals = pd.read_parquet("./output_quantity_cut_1.pq")['redshift'].index.tolist()

    pdfs_file = "../paper_data/GPz_GL/"+deg+"/output_estimate_GPz.hdf5"

    est_ens = qp.read(pdfs_file)
    truth_ens = qp.read(true_posts_file)

    xvals = h5py.File(true_posts_file)['meta']['xvals'][0]
    
    cvm_ls = []
    ks_ls = []
    rmse_ls = []
    kld_ls = []

    fails = []
    

    for i in range(0, len(gals)-1):
        if i%1000==0:
            print(f'processing gal {i}')

        # the true posterior is the array-like first argument
        truth = h5py.File(true_posts_file)['data']['yvals'][i].tolist()

        if np.isnan(h5py.File(pdfs_file)['data']['loc'][i])==True or np.isnan(h5py.File(pdfs_file)['data']['scale'][i])==True:
            fails.append(gals[i])
            print(deg, gals[i])  

            cvm_ls.append(np.nan)
            ks_ls.append(np.nan)
            rmse_ls.append(np.nan)
            kld_ls.append(np.nan)

        elif np.isnan(truth[0])==True:
            cvm_ls.append(np.nan)
            ks_ls.append(np.nan)
            rmse_ls.append(np.nan)
            kld_ls.append(np.nan)
            pass

        else:

            cvm = scipy.stats.cramervonmises(truth_ens[i].rvs(size = 1000)[0], est_ens[i].cdf)
            cvm_ls.append(cvm.statistic)
#             print('cvm for '+spec+' done')

            ks = scipy.stats.kstest(truth_ens[i].rvs(size = 1000)[0], est_ens[i].cdf(xvals)[0])
            ks_ls.append(ks.statistic)
            #print('ks for '+spec+' done')

            rmse = sklearn.metrics.mean_squared_error(truth_ens[i].pdf(xvals)[0], est_ens[i].pdf(xvals)[0])
            rmse_ls.append(rmse)
            #print('rmse for '+spec+' done')

            truth_data = truth_ens[i].pdf(xvals)[0]
            est_data = est_ens[i].pdf(xvals)[0] 

            truth_data[truth_data < 1e-300] = 1e-300
            est_data[est_data < 1e-300] = 1e-300

            kld = scipy.special.kl_div(truth_data, est_data)
            kld_ls.append(np.sum(kld) * 3.0 / len(gals))
            # print('kld for '+spec+' done')
    
    GPz_GL_cvm_dict[deg] = cvm_ls
    GPz_GL_ks_dict[deg] = ks_ls
    GPz_GL_rmse_dict[deg] = rmse_ls
    GPz_GL_kld_dict[deg] = kld_ls
    print(deg+' done')
    

In [ ]:
for deg in deg_ls: 
    getGPzGLDistStats(deg)

In [ ]:
GPz_GL_kld_dict.pop('LSSTerr_null')

In [ ]:
writeToFiles_tq('GPz', GPz_GL_cvm_dict, GPz_GL_ks_dict, GPz_GL_rmse_dict, GPz_GL_kld_dict)

In [ ]:
# cvm = pd.read_parquet('../paper_data/GPz_GL/cvm_stats.pq')
# cvm

In [ ]:
# ks = pd.read_parquet('../paper_data/GPz_GL/ks_stats.pq')
# ks

In [ ]:
# rmse = pd.read_parquet('../paper_data/GPz_GL/rmse_stats.pq')
# rmse

In [ ]:
# kld = pd.read_parquet('../paper_data/GPz_GL/kld_stats.pq')
# kld

In [ ]:
print(np.log(kld['BOSS']))

In [ ]:
#plt.hist(np.log(kld['LSSTerr_only']), bins = np.linspace(-10, 1, 100), alpha = 0.5, label = 'none')
plt.hist(np.log(kld['invz=0.1']), bins = np.linspace(-10, 1, 100), alpha = 0.5, label = '0.1')
plt.hist(np.log(kld['LSSTerr_only']), bins = np.linspace(-10, 1, 100), alpha = 0.5, label = '1.0')
plt.legend()

### Not GL

In [ ]:
GPz_cvm_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             'control': []
             }

In [ ]:
GPz_ks_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             'control': []
             }

In [ ]:
GPz_rmse_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             'control': []
             }

In [ ]:
GPz_kld_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             'control': []
             }

In [ ]:
GPz_fails_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             'control': []
             }

In [ ]:
G_ks = {'invz=0.3': []}
G_cvm = {'invz=0.3': []}
G_rmse = {'invz=0.3': []}
G_kld = {'invz=0.3': []}

In [ ]:
true_posts_file = "../paper_data/output_lsstErr_test_data_posts.hdf5"
gals = pd.read_parquet("../paper_data/output_quantity_cut_1.pq")['redshift'].index.tolist()

for i in range(0, len(gals)):
    h5py.File(true_posts_file)['data']['yvals'][i].tolist()


In [ ]:
def getGPzDistStats(deg):

    true_posts_file = "../paper_data/output_lsstErr_test_data_posts.hdf5"
    gals = pd.read_parquet("../paper_data/output_quantity_cut_1.pq")['redshift'].index.tolist()

    os.chdir("../paper_data/GPz/"+deg)
    pdfs_file = "../paper_data/GPz/"+deg+"/output_estimate_GPz.hdf5"

    est_ens = qp.read(pdfs_file)
    truth_ens = qp.read(true_posts_file)

    xvals = h5py.File(true_posts_file)['meta']['xvals'][0]
    
    cvm_ls = []
    ks_ls = []
    rmse_ls = []
    kld_ls = []

    fails = []
    

    for i in range(0, len(gals)-1):

        # the true posterior is the array-like first argument
        truth = h5py.File(true_posts_file)['data']['yvals'][i].tolist()

        if np.isnan(h5py.File(pdfs_file)['data']['loc'][i])==True or np.isnan(h5py.File(pdfs_file)['data']['scale'][i])==True:
            fails.append(gals[i])
            print(deg, gals[i])  

            cvm_ls.append(np.nan)
            ks_ls.append(np.nan)
            rmse_ls.append(np.nan)
            kld_ls.append(np.nan)

        elif np.isnan(truth[0])==True:
            cvm_ls.append(np.nan)
            ks_ls.append(np.nan)
            rmse_ls.append(np.nan)
            kld_ls.append(np.nan)
            pass

        else:

            cvm = scipy.stats.cramervonmises(truth_ens[i].rvs(size = 1000)[0], est_ens[i].cdf)
            cvm_ls.append(cvm.statistic)
#             print('cvm for '+spec+' done')

            ks = scipy.stats.kstest(truth_ens[i].rvs(size = 1000)[0], est_ens[i].cdf(xvals)[0])
            ks_ls.append(ks.statistic)
            #print('ks for '+spec+' done')

            rmse = sklearn.metrics.mean_squared_error(truth_ens[i].pdf(xvals)[0], est_ens[i].pdf(xvals)[0])
            rmse_ls.append(rmse)
            #print('rmse for '+spec+' done')

            truth_data = truth_ens[i].pdf(xvals)[0]
            est_data = est_ens[i].pdf(xvals)[0] 

            truth_data[truth_data < 1e-300] = 1e-300
            est_data[est_data < 1e-300] = 1e-300

            kld = scipy.special.kl_div(truth_data, est_data)
            kld_ls.append(np.sum(kld) * 3.0 / len(gals))
            # print('kld for '+spec+' done')
    
    
    # GPz_cvm_dict[deg] = cvm_ls
    # GPz_ks_dict[deg]=ks_ls
    # GPz_rmse_dict[deg]=rmse_ls
    #GPz_kld_dict[deg] = kld_ls
    #GPz_fails_dict[deg] = fails

    G_ks[deg] = ks_ls
    G_cvm[deg] = cvm_ls
    G_rmse[deg] = rmse_ls
    G_kld[deg] = kld_ls
    print(deg+" done")

# CMNN_dict['DEEP2']
    

In [ ]:
for deg in extra_invz:
    getGPzDistStats(deg)

In [ ]:
G_df_tmp = pd.read_parquet('../paper_data/GPz/cvm_stats.pq')
G_df = G_df_tmp#.drop(columns='control') 
G_df.insert(8, 'invz=0.3', G_cvm['invz=0.3'])
#G_df.insert(8, 'invz=0.6', G_kld['invz=0.6'])
G_df.to_parquet('../paper_data/GPz/cvm_stats.pq')
G_df

In [ ]:
for deg in deg_ls: 
    getGPzDistStats(deg)

In [ ]:
writeToFiles('GPz', GPz_kld_dict)#GPz_cvm_dict, GPz_ks_dict, GPz_rmse_dict, GPz_kld_dict)

In [ ]:
pd.read_parquet('../paper_data/GPz/kld_stats.pq')

## FZBoost

In [ ]:
FZBoost_cvm_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'control': []
             }

In [ ]:
FZBoost_ks_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'control': []
             }

In [ ]:
FZBoost_rmse_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'control': []
             }

In [ ]:
FZBoost_kld_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'control': []
             }

In [ ]:
FZBoost_fails_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'control': []
             }

In [ ]:
F_ks = {'invz=0.3': []}
F_cvm = {'invz=0.3': []}
F_rmse = {'invz=0.3': []}
F_kld = {'invz=0.3': []}

In [ ]:
def getFZBoostDistStats(deg):

    true_posts_file = "../paper_data/output_lsstErr_test_data_posts.hdf5"
    gals = pd.read_parquet("./output_quantity_cut_1.pq")['redshift'].index.tolist()

#     os.chdir("../paper_data/FZBoost/"+deg)
    pdfs_file = "../paper_data/FZBoost/"+deg+"/output_estimate_FZBoost.hdf5"

    truth_ens = qp.read(true_posts_file)
    est_ens = qp.read(pdfs_file)

    xvals = h5py.File(true_posts_file)['meta']['xvals'][0]
    
    cvm_ls = []
    ks_ls = []
    rmse_ls = []
    kld_ls = []

    fails = []
    
    for i in range(0, len(gals)-1):
        if i%1000==0:
            print(f'processing gal {i}')
        # the true posterior is the array-like first argument
        truth = h5py.File(true_posts_file)['data']['yvals'][i].tolist()
        
        yvals = h5py.File(pdfs_file)['data']['yvals'][i].tolist()

        if np.isnan(yvals[0])==True:
            fails.append(gals[i])
            print(deg, gals[i])  

            cvm_ls.append(np.nan)
            ks_ls.append(np.nan)
            rmse_ls.append(np.nan)
            kld_ls.append(np.nan)

        elif np.isnan(truth[0])==True:
            cvm_ls.append(np.nan)
            ks_ls.append(np.nan)
            rmse_ls.append(np.nan)
            kld_ls.append(np.nan)
            pass

        else:      
            cvm = scipy.stats.cramervonmises(truth_ens[i].rvs(size = 1000)[0], est_ens[i].cdf)
            cvm_ls.append(cvm.statistic)
#             print('cvm for '+spec+' done')

            ks = scipy.stats.kstest(truth_ens[i].rvs(size = 1000)[0], est_ens[i].cdf(xvals)[0])
            ks_ls.append(ks.statistic)
            #print('ks for '+spec+' done')

            rmse = sklearn.metrics.mean_squared_error(truth_ens[i].pdf(xvals)[0], est_ens[i].pdf(xvals)[0])
            rmse_ls.append(rmse)
            #print('rmse for '+spec+' done')


            truth_data = truth_ens[i].pdf(xvals)[0]
            est_data = est_ens[i].pdf(xvals)[0] 

            truth_data[truth_data == 0.0] = 1e-300
            est_data[est_data == 0.0] = 1e-300

            kld = scipy.special.kl_div(truth_data, est_data)
            kld_ls.append(np.sum(kld) * 3.0 / len(gals))
            # print('kld for '+spec+' done')

        
    # FZBoost_cvm_dict[deg] = cvm_ls
    # FZBoost_ks_dict[deg]=ks_ls
    # FZBoost_rmse_dict[deg]=rmse_ls
    # FZBoost_kld_dict[deg] = kld_ls
    #FZBoost_fails_dict[deg] = fails

    F_ks[deg] = ks_ls
    F_cvm[deg] = cvm_ls
    F_rmse[deg] = rmse_ls
    F_kld[deg] = kld_ls

    print(deg+' done')


In [ ]:
extra_invz

In [ ]:
for deg in extra_invz:
    getFZBoostDistStats(deg)

In [ ]:
# F_df_tmp = pd.read_parquet('../paper_data/FZBoost/cvm_stats.pq')
# F_df = F_df_tmp#.drop(columns='control') 
# #F_df.insert(7, 'invz=0.1', F_kld['invz=0.1'])
# F_df.insert(8, 'invz=0.3', F_cvm['invz=0.3'])
# F_df.to_parquet('../paper_data/FZBoost/cvm_stats.pq')
# F_df

In [ ]:
for deg in deg_ls: 
    getFZBoostDistStats(deg)

In [ ]:
writeToFiles_tq('FZBoost', F_cvm, F_ks, F_rmse, F_kld)

In [ ]:
pd.read_parquet('../paper_data/FZBoost/kld_stats.pq')

## PZFlow

In [ ]:
PZFlow_cvm_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [], 
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             #'control': []
             }

In [ ]:
PZFlow_ks_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [], 
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             #'control': []
             }

In [ ]:
PZFlow_rmse_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [], 
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             #'control': []
             }

In [ ]:
PZFlow_kld_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [], 
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             #'control': []
             }

In [ ]:
PZFlow_fails_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [], 
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             #'control': []
             }

In [ ]:
P_ks = {'invz=0.3': []}
P_cvm = {'invz=0.3': []}
P_rmse = {'invz=0.3': []}
P_kld = {'invz=0.3': []}

In [ ]:
def getPZFlowDistStats(deg):
      
    true_posts_file = "../paper_data/output_lsstErr_test_data_posts.hdf5"
    gals = pd.read_parquet("./output_quantity_cut_1.pq")['redshift'].index.tolist()

    truth_ens = qp.read(true_posts_file)
    xvals = h5py.File(true_posts_file)['meta']['xvals'][0]


#     os.chdir("../paper_data/PZFlow/"+deg)

    pdfs_file = "../paper_data/PZFlow/"+deg+"/output_estimate_pzflow.hdf5"
    est_ens = qp.read(pdfs_file)

    cvm_ls = []
    ks_ls = []
    rmse_ls = []
    kld_ls = []

    fails = []
    
    for i in range(0, len(gals)-1):
        if i%1000==0:
            print(f'processing gal {i}')

        # the true posterior is the array-like first argument
        truth = h5py.File(true_posts_file)['data']['yvals'][i].tolist()

        yvals = h5py.File(pdfs_file)['data']['yvals'][i].tolist()

        if np.isnan(yvals[0])==True:
            fails.append(gals[i])
            cvm_ls.append(np.nan)
            ks_ls.append(np.nan)
            rmse_ls.append(np.nan)
            kld_ls.append(np.nan)

        elif np.isnan(truth[0])==True:
            cvm_ls.append(np.nan)
            ks_ls.append(np.nan)
            rmse_ls.append(np.nan)
            kld_ls.append(np.nan)
            pass
        
        else:      
            cvm = scipy.stats.cramervonmises(truth_ens[i].rvs(size = 1000)[0], est_ens[i].cdf)
            cvm_ls.append(cvm.statistic)
#             print('cvm for '+spec+' done')

            ks = scipy.stats.kstest(truth_ens[i].rvs(size = 1000)[0], est_ens[i].cdf(xvals)[0])
            ks_ls.append(ks.statistic)
            #print('ks for '+spec+' done')

            rmse = sklearn.metrics.mean_squared_error(truth_ens[i].pdf(xvals)[0], est_ens[i].pdf(xvals)[0])
            rmse_ls.append(rmse)
            #print('rmse for '+spec+' done')

            truth_data = truth_ens[i].pdf(xvals)[0]
            est_data = est_ens[i].pdf(xvals)[0] 

            truth_data[truth_data == 0.0] = 1e-300
            est_data[est_data == 0.0] = 1e-300

            kld = scipy.special.kl_div(truth_data, est_data)
            kld_ls.append(np.sum(kld) * 3.0 / len(gals))
            # print('kld for '+spec+' done')

    # PZFlow_cvm_dict[deg] = cvm_ls
    # PZFlow_ks_dict[deg]=ks_ls
    # PZFlow_rmse_dict[deg]=rmse_ls
    #PZFlow_kld_dict[deg] = kld_ls
    #PZFlow_fails_dict[deg] = fails

    P_ks[deg] = ks_ls
    P_cvm[deg] = cvm_ls
    P_rmse[deg] = rmse_ls
    P_kld[deg] = kld_ls

    print(deg+' done')
    return([ks_ls, cvm_ls, rmse_ls, kld_ls])
    


In [ ]:
PGama = getPZFlowDistStats('GAMA')

In [ ]:
Pks = pd.read_parquet('../paper_data/PZFlow/ks_stats.pq')
Pcvm = pd.read_parquet('../paper_data/PZFlow/cvm_stats.pq')
Prmse = pd.read_parquet('../paper_data/PZFlow/rmse_stats.pq')
Pkld = pd.read_parquet('../paper_data/PZFlow/kld_stats.pq')

Pks['GAMA'] = PGama[0]
Pcvm['GAMA'] = PGama[0]
Prmse['GAMA'] = PGama[0]
Pkld['GAMA'] = PGama[0]

In [ ]:
Pks.to_parquet('../paper_data/PZFlow/ks_stats.pq')
Pcvm.to_parquet('../paper_data/PZFlow/cvm_stats.pq')
Prmse.to_parquet('../paper_data/PZFlow/rmse_stats.pq')
Pkld.to_parquet('../paper_data/PZFlow/kld_stats.pq')

In [ ]:
for deg in extra_invz:
    getPZFlowDistStats(deg)

In [ ]:
P_df_tmp = pd.read_parquet('../paper_data/PZFlow/cvm_stats.pq')
P_df = P_df_tmp#.drop(columns='control') 
#P_df.insert(7, 'invz=0.1', P_kld['invz=0.1'])
P_df.insert(8, 'invz=0.3', P_cvm['invz=0.3'])
P_df.to_parquet('../paper_data/PZFlow/cvm_stats.pq')
P_df

In [ ]:
deg_ls

In [ ]:
for deg in deg_ls: 
    getPZFlowDistStats(deg)

In [ ]:


# #PZFlow_cvm_dict['GAMA'] = PZFlow_cvm_dict['GAMA'].

# PZFlow_cvm_dict['GAMA'] = np.nan*np.ones(31247)
# PZFlow_ks_dict['GAMA'] = np.nan*np.ones(31247)
# PZFlow_rmse_dict['GAMA'] = np.nan*np.ones(31247)
# PZFlow_kld_dict['GAMA'] = np.nan*np.ones(31247)

In [ ]:
PZFlow_cvm_dict

In [ ]:
# writeToFiles_tq('PZFlow', PZFlow_cvm_dict, PZFlow_ks_dict, PZFlow_rmse_dict, PZFlow_kld_dict)

In [ ]:
writeToFiles_tq('PZFlow', P_cvm, P_ks, P_rmse, P_kld)

In [ ]:
getPZFlowDistStats('control')

In [ ]:
cvm_df = pd.DataFrame.from_dict(PZFlow_control_cvm_dict, orient = 'columns')
cvm_path = '../paper_data/PZFlow/control_cvm_stats.pq'
cvm_df.to_parquet(cvm_path)

ks_df = pd.DataFrame.from_dict(PZFlow_control_ks_dict, orient = 'columns')
ks_path = '../paper_data/PZFlow/control_ks_stats.pq'
ks_df.to_parquet(ks_path)

rmse_df = pd.DataFrame.from_dict(PZFlow_control_rmse_dict, orient = 'columns')
rmse_path = '../paper_data/PZFlow/control_rmse_stats.pq'
rmse_df.to_parquet(rmse_path)

kld_df = pd.DataFrame.from_dict(PZFlow_control_kld_dict, orient = 'columns')
kld_path = '../paper_data/PZFlow/control_kld_stats.pq'
kld_df.to_parquet(kld_path)


# Dist-to-dist plots

In [ ]:
import numpy as np

def removeNans(df, cols):
    ls = []
    for col in cols: 
        ls +=  df.index[np.isnan(df[col]) ==True].tolist()

    drops = list(set(ls))
    new_df = df.drop(drops)

    return new_df 

deg_ls2 = ['BOSS', 'DEEP2', 'GAMA', 'HSC', 'VVDSf02', 'zCOSMOS', 'invz=1.0', 'invz=1.4', 'LSSTerr_only', 'LSSTerr_null']

In [ ]:
df = pd.read_parquet('../paper_data/CMNN/ks_stats.pq')

df['BOSS'].index[]
# 


In [ ]:
def removeInfs(df, col): 
    ls = []
    ls +=  df.index[np.isfinite(df[col]) ==False].tolist()

    new_df = df.drop(ls)

    return new_df[col]

In [ ]:
file = pd.read_parquet('../paper_data/CMNN/kld_stats.pq')

removeInfs(file, 'DEEP2').tolist()

In [ ]:
def makeStatHists(stat, deg, min, max, binnum, axes):

    import matplotlib.pyplot as plt
    import pandas as pd 
    
    grid = np.linspace(min, max, binnum)

    t = removeInfs(pd.read_parquet('../paper_data/TrainZ/'+stat+'_stats.pq'), deg ).tolist()
    c = removeInfs(pd.read_parquet('../paper_data/CMNN/'+stat+'_stats.pq'),  deg ).tolist()
    g = removeInfs(pd.read_parquet('../paper_data/GPz/'+stat+'_stats.pq'),  deg ).tolist()
    f = removeInfs(pd.read_parquet('../paper_data/FZBoost/'+stat+'_stats.pq'),  deg ).tolist()

    axes.hist(t, label = 'TrainZ', bins = grid, alpha = 0.5)
    axes.hist(c, label = 'CMNN', bins = grid, alpha = 0.5)
    axes.hist(g, label = 'GPz', bins = grid, alpha = 0.5)
    axes.hist(f, label = 'FZBoost', bins = grid, alpha = 0.5)
    
    if deg != 'control':
        p = removeInfs(pd.read_parquet('../paper_data/PZFlow/'+stat+'_stats.pq'), deg ).tolist()
        axes.hist(p, label = 'PZFlow', bins = grid, alpha = 0.5)

    elif deg == 'control':
        pcontrol = removeInfs(pd.read_parquet('../paper_data/PZFlow/control_'+stat+'_stats.pq'),  deg ).tolist()
        axes.hist(pcontrol, label = 'PZFlow', bins = grid, alpha = 0.5)
        
    axes.legend()
    axes.set_xlim(min, max)
    #axes.ylim(0, 10000)
    axes.set_xlabel("Statistic Value")
    axes.set_ylabel("Bin Count")
    
    axes.set_title(stat)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

trainset = 'control'

fig, axes = plt.subplots(nrows = 2, ncols = 2, figsize = (24, 16) )


makeStatHists('ks', trainset, 0, 1, 100, axes[0][0])
makeStatHists('cvm', trainset, 0, 35, 100, axes[0][1])
makeStatHists('rmse', trainset, 0, 20, 100, axes[1][0])
makeStatHists('kld', trainset, 2.19, 2.23, 100, axes[1][1])

fig.suptitle("Distribution of statistic values for control training set", size = 'xx-large')

In [ ]:
frame = pd.read_parquet('../paper_data/PZFlow/kld_stats.pq')['invz=1.0']

ct = 0
for obj in frame: 
    if np.isfinite(obj)==False:
        ct += 1

print(ct)

In [ ]:
gdf = pd.read_parquet('../paper_data/GPz/kld_stats.pq')
cmdf = pd.read_parquet('../paper_data/CMNN/kld_stats.pq')

plt.scatter(gdf['redshift'], gdf['control'], s=0.5)
plt.scatter(cmdf['redshift'], cmdf['control'], s=0.5)

In [ ]:
gdf1 = pd.read_parquet('../paper_data/GPz/modes.pq')
cmdf1 = pd.read_parquet('../paper_data/CMNN/modes.pq')

plt.scatter(gdf1['redshift'], gdf1['control'], s=0.5)
#plt.scatter(cmdf1['redshift'], cmdf1['control'], s=0.5)

In [ ]:
gpzfile = '../paper_data/GPz/BOSS/output_estimate_GPz.hdf5'
cmnnfile = '../paper_data/CMNN/BOSS/output_estimate_CMNN.hdf5'
truthfile = '../paper_data/output_lsstErr_test_data_posts.hdf5'
points = pd.read_parquet('../paper_data/output_quantity_cut_1.pq')['redshift'].tolist()

gpzens = qp.read(gpzfile)
cmnnens = qp.read(cmnnfile)
truthens = qp.read(truthfile)

grid = np.linspace(0, 100, 301)

gal = 6
plt.plot(grid, gpzens[gal].pdf(grid)[0])
#plt.plot(grid, cmnnens[gal].pdf(grid)[0])
#plt.plot(grid, truthens[gal].pdf(grid)[0], color = 'k')
#plt.plot(points[gal]*np.ones(301), np.linspace(0, 30, 301), color = 'k', linestyle = 'dashed')


l = h5py.File(gpzfile)['data']['loc'][gal]
s = h5py.File(gpzfile)['data']['scale'][gal]
print(l, s)

# Failiure Rates

In [ ]:
df = pd.read_parquet('../paper_data/PZFlow/ks_stats.pq')
df

In [ ]:
plt.scatter(df['redshift'], df['LSSTerr_only'], s = 0.1)
plt.xlabel('redshift')
plt.ylabel('statistic value')

# Mode Comparisons

In [ ]:
deg_ls = ['BOSS', 
          'DEEP2', 
          'GAMA', 
          'HSC', 
          'VVDSf02', 
          'zCOSMOS', 
          'invz=0.1',
          'invz=0.3', 
          'invz=0.6',
          'invz=1.0', 
          'invz=1.4', 
          'LSSTerr_only', 
          'LSSTerr_null'
          ]

In [ ]:
def getModes(filepath):

    file = h5py.File(filepath)

    if 'loc' in file['data'].keys():
        modes = []
        for loc in file['data']['loc']: 
            modes.append(loc[0])

        print(deg+' done')
        return modes
    
    else: 
        inds = []
        modes = []
        
        xvals = file['meta']['xvals'][0]
        yvals = file['data']['yvals']

        for data in yvals: 
            if np.isnan(data[0])==False:
                data = data.tolist()
                max = np.max(data)
                ind = data.index(max)
                modes.append(xvals[ind])
            else:
                modes.append(np.nan)
        
        for ind in inds: 
            modes.append(xvals[ind])
        
        print(deg+' done')
        
        return modes


In [ ]:
print(getModes('../paper_data/GPz/DEEP2/output_estimate_gpz.hdf5'))

In [ ]:
GPz_GL_mode_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [], 
             'invz=0.3': [], 
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             #'LSSTerr_null': [], 
             #'control': []
             }


In [ ]:
for deg in deg_ls: 
    GPz_GL_mode_dict[deg] = getModes('../paper_data/GPz_GL/'+deg+'/output_estimate_GPz.hdf5')

In [ ]:
redshift = pd.read_parquet('../paper_data/TrainZ/kld_stats.pq')['redshift']
len(redshift)

In [ ]:


# GPz_GL_modes = pd.DataFrame.from_dict(GPz_GL_mode_dict, orient='columns').drop(31247)
# #GPz_GL_modes.insert(0, 'redshift', redshift)
# GPz_GL_modes.to_parquet('../paper_data/GPz_GL/modes.pq')
# pd.read_parquet('../paper_data/GPz_GL/modes.pq')

In [ ]:
TrainZ_mode_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [], 
             'invz=0.3': [], 
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             #'control': []
             }

CMNN_mode_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [], 
             'invz=0.3': [], 
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             #'control': []
             }
GPz_mode_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [], 
             'invz=0.3': [], 
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             #'control': []
             }

FZBoost_mode_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [], 
             'invz=0.3': [], 
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             #'control': []
             }

PZFlow_mode_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [], 
             'invz=0.3': [], 
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             #'control': []
             }

In [ ]:
est_ls = ['TrainZ', 
          'CMNN', 
          'GPz', 
          'FZBoost', 
          'PZFlow']

In [ ]:
for deg in deg_ls: 
    TrainZ_mode_dict[deg] = getModes('../paper_data/TrainZ/'+deg+'/output_estimate_TrainZ.hdf5')

In [ ]:
for deg in deg_ls: 
    CMNN_mode_dict[deg] = getModes('../paper_data/CMNN/'+deg+'/output_estimate_CMNN.hdf5')

In [ ]:
for deg in deg_ls: 
    GPz_mode_dict[deg] = getModes('../paper_data/GPz/'+deg+'/output_estimate_GPz.hdf5')

In [ ]:
for deg in deg_ls: 
    FZBoost_mode_dict[deg] = getModes('../paper_data/FZBoost/'+deg+'/output_estimate_FZBoost.hdf5')

In [ ]:
for deg in deg_ls: 
    PZFlow_mode_dict[deg] = getModes('../paper_data/PZFlow/'+deg+'/output_estimate_pzflow.hdf5')

In [ ]:
redshift = pd.read_parquet('../paper_data/TrainZ/kld_stats.pq')['redshift']

In [ ]:
# TrainZ_modes = pd.DataFrame.from_dict(TrainZ_mode_dict, orient='columns').drop(31247)
# TrainZ_modes.insert(0, 'redshift', redshift)
# TrainZ_modes.to_parquet('../paper_data/TrainZ/modes.pq')
# TrainZ_modes

In [ ]:
# CMNN_modes = pd.DataFrame.from_dict(CMNN_mode_dict, orient='columns').drop(31247)
# CMNN_modes.insert(0, 'redshift', redshift)
# CMNN_modes.to_parquet('../paper_data/CMNN/modes.pq')
# CMNN_modes

In [ ]:
# GPz_modes = pd.DataFrame.from_dict(GPz_mode_dict, orient='columns').drop(31247)
# GPz_modes.insert(0, 'redshift', redshift)
# GPz_modes.to_parquet('../paper_data/GPz/modes.pq')
# GPz_modes

In [ ]:
# FZBoost_modes = pd.DataFrame.from_dict(FZBoost_mode_dict, orient='columns').drop(31247)
# FZBoost_modes.insert(0, 'redshift', redshift)
# FZBoost_modes.to_parquet('../paper_data/FZBoost/modes.pq')
# FZBoost_modes

In [ ]:
# PZFlow_modes = pd.DataFrame.from_dict(PZFlow_mode_dict, orient='columns').drop(31247)

# PZFlow_modes.insert(0, 'redshift', redshift)

# PZFlow_modes.to_parquet('../paper_data/PZFlow/modes.pq')
# PZFlow_modes

In [ ]:
df = pd.read_parquet('../paper_data/GPz_GL/modes.pq')

line = np.linspace(0, 3, 101)
plt.scatter(df['redshift'], df['LSSTerr_only'], s=0.1)
#plt.scatter(df['redshift'], df['invz=1.4'], s=0.1)
plt.plot(line, line, color = 'k', linestyle = 'dashed')
plt.xlabel('redshift')
plt.ylabel('estimator output')


In [ ]:
df = pd.read_parquet('../paper_data/GPz/modes.pq')

line = np.linspace(0, 3, 101)
plt.scatter(df['redshift'], df['control'], s=0.1)
#plt.scatter(df['redshift'], df['invz=1.4'], s=0.1)
plt.plot(line, line, color = 'k', linestyle = 'dashed')
plt.xlabel('redshift')
plt.ylabel('estimator output')


In [ ]:
df = pd.read_parquet('../paper_data/FZBoost/modes.pq')

line = np.linspace(0, 3, 101)
plt.scatter(df['redshift'], df['control'], s=0.1)
#plt.scatter(df['redshift'], df['invz=1.4'], s=0.1)
plt.plot(line, line, color = 'k', linestyle = 'dashed')
plt.xlabel('redshift')
plt.ylabel('estimator output')


In [ ]:
df = pd.read_parquet('../paper_data/PZFlow/control_modes.pq')

line = np.linspace(0, 3, 101)
plt.scatter(df['redshift'], df['control'], s=0.1)
plt.plot(line, line, color = 'k', linestyle = 'dashed')
plt.show()
#plt.scatter(df['redshift'], df['VVDSf02'], s=0.1)


In [ ]:
df = pd.read_parquet('../paper_data/TrainZ/modes.pq')

line = np.linspace(0, 3, 101)
plt.scatter(df['redshift'], df['control'], s=0.1)
plt.plot(line, line, color = 'k', linestyle = 'dashed')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

deg = 'LSSTerr_only'

fig, axes = plt.subplots(nrows = 2, ncols = 2, figsize = (15, 15)) 

df1 = pd.read_parquet('../paper_data/CMNN/modes.pq')
df2 = pd.read_parquet('../paper_data/GPz_GL/modes.pq')
df3 = pd.read_parquet('../paper_data/FZBoost/modes.pq')
df4 = pd.read_parquet('../paper_data/PZFlow/modes.pq')
#df4c = pd.read_parquet('../paper_data/PZFlow/control_modes.pq')

line = np.linspace(1.5, 3, 101)

m = axes[0][0].hist2d(df1['redshift'], df1[deg], bins = 100, cmin = 1, vmin = 1, vmax = 550)#s=0.5)
axes[0][0].plot(line, line, color = 'orange', linestyle = 'dashed', alpha = 0.5)
axes[0][0].set_title('CMNN')
axes[0][0].set_xlim(0, 3)
#axes[0][0].set_ylim(0, 3)

n = axes[0][1].hist2d(df2['redshift'], df2[deg], bins = 100, cmin = 1, vmin = 1, vmax = 550)#s=0.5)
axes[0][1].plot(line, line, color = 'orange', linestyle = 'dashed', alpha = 0.5)
axes[0][1].set_title('GPz')
axes[0][1].set_xlim(0, 3)
#axes[0][1].set_ylim(0, 3)

h = axes[1][0].hist2d(df3['redshift'], df3[deg], bins = 100, cmin = 1, vmin = 1, vmax = 550)#s=0.5)
axes[1][0].plot(line, line, color = 'orange', linestyle = 'dashed', alpha = 0.5)
axes[1][0].set_title('FZBoost')
axes[1][0].set_xlim(0, 3)
#axes[1][0].set_ylim(0, 3)

if deg =='control':
    g = axes[1][1].hist2d(df4c['redshift'], df4c[deg], bins = 100, cmin = 1, vmin = 1, cmax = 550)#s=0.5)
else: 
    inds = []
    z = []
    data = []
    pzfdata = df4[deg].tolist()
    ct = 0
    for i in pzfdata:
        if np.isnan(i)==False: 
            data.append(i)
            inds.append(ct)
        ct += 1

    ct2 = 0
    for j in df4['redshift']:
        if ct2 in inds: 
            z.append(j)
        ct2 += 1
    
    axes[1][1].hist2d(z, data, bins = 100, cmin = 1, vmin = 1, vmax = 550)
axes[1][1].plot(line, line, color = 'orange', linestyle = 'dashed', alpha=0.5)
axes[1][1].set_title('PZFlow')
axes[1][1].set_xlim(0, 3)
#axes[1][1].set_ylim(0, 3)


fig.suptitle("Mode point estimates for LSST error model only training set degradation", size = 'x-large')
fig.supxlabel('Spectroscopic redshift')
fig.supylabel('Mode of photometric redshift')
fig.colorbar(n[3], ax=axes)


# Outlier Rejection

In [ ]:
import numpy as np
import scipy 
from scipy import stats

def outlierRejection(ls):
    # for i in ls: 
    #     if np.isnan(i)==True:
    #         ls.remove(i)
    mean = np.mean(ls)
    stdev = stats.tstd(ls)
    #print(mean, stdev)
    len_b4 = len(ls)
    for elem in ls:
        if np.abs(mean - elem) > 3*stdev:
            ls.remove(elem) 
    len_aft = len(ls)
    #print(len_b4, len_aft)
    if np.abs(len_b4 - len_aft) > 0:
        return outlierRejection(ls)
    else:
        print(mean, stdev)
        return ls

In [ ]:
import pandas as pd

def distanceOutlierRejection(est, deg): 
    df = pd.read_parquet('../paper_data/'+est+'/modes.pq')
    data = []
    z = []
    for i in range(len(df[deg])):
        if np.isnan(df[deg][i])==False:
            data.append(df[deg][i])
            z.append(df['redshift'][i])
    ls = []
    for j in range(len(data)):
        ls.append(np.abs(z[j] - data[j]) / np.sqrt(2))
    return [outlierRejection(ls), str(( len(z) - len(outlierRejection(ls)) ) / len(z) * 100)[0:5]+'%']

ans = distanceOutlierRejection('GPz', 'VVDSf02')[1]


In [ ]:
three_dict = {'TrainZ':[], 
                'CMNN': [], 
                'GPz_GL': [], 
                'FZBoost': [], 
                'PZFlow': []}
invz_ls = [
          'invz=0.1',
          'invz=0.3', 
          'invz=0.6',
          'invz=1.0', 
          'invz=1.4', 
          'LSSTerr_only', ]

spec_ls = ['BOSS', 
          'DEEP2', 
          'GAMA', 
          'HSC', 
          'VVDSf02', 
          'zCOSMOS', ] 


for key in three_dict: 
    tmp = []
    for pivot in invz_ls: 
        print(distanceOutlierRejection(key, pivot)[1])
        tmp.append(distanceOutlierRejection(key, pivot)[1])
    three_dict[key] = tmp


In [ ]:
threedf = pd.DataFrame.from_dict(three_dict)
threedf.to_parquet('../paper_data/3sigma_invz_outliers.pq')

In [ ]:
threedf

In [ ]:
pivots = [0.1, 0.3, 0.6, 1.0, 1.4]

In [ ]:
spec_three_dict = {'TrainZ':[], 
                'CMNN': [], 
                'GPz_GL': [], 
                'FZBoost': [], 
                'PZFlow': []}

spec_ls = ['BOSS', 
          'DEEP2', 
          'GAMA', 
          'HSC', 
          'VVDSf02', 
          'zCOSMOS',
          'LSSTerr_only', ]


for key in spec_three_dict: 
    tmp = []
    for spec in spec_ls: 
        print(distanceOutlierRejection(key, spec)[1])
        tmp.append(distanceOutlierRejection(key, spec)[1])
    spec_three_dict[key] = tmp

In [ ]:
specthreedf = pd.DataFrame.from_dict(spec_three_dict)
specthreedf.to_parquet('../paper_data/3sigma_spec_outliers.pq')

In [ ]:
specthreedf

In [ ]:
deg_ls = ['BOSS', 
          'DEEP2', 
          'GAMA', 
          'HSC', 
          'VVDSf02', 
          'zCOSMOS', 
          'invz=0.1',
          'invz=0.3', 
          'invz=0.6',
          'invz=1.0', 
          'invz=1.4', 
          'LSSTerr_only', 
          #'LSSTerr_null', 
          ]

In [ ]:
outlier_dict = {'TrainZ':[], 
                'CMNN': [], 
                'GPz_GL': [], 
                'FZBoost': [], 
                'PZFlow': []}

In [ ]:
GL_outlier_dict = {'GPz_GL': []}

In [ ]:
for key in outlier_dict:
    ls = []
    for deg in deg_ls: 
        ls.append(distanceOutlierRejection(key, deg)[1])
    print(ls)
    outlier_dict[key] = ls

In [ ]:
ORdf = pd.DataFrame.from_dict(outlier_dict)
ORdf.to_parquet()
ORdf

In [ ]:
out_df = pd.read_parquet('../paper_data/outlier_rejection.pq')

out_df['GPz_GL'] = GL_outlier_dict['GPz_GL'] 
out_df.to_parquet('../paper_data/outlier_rejection.pq')
out_df = pd.read_parquet('../paper_data/outlier_rejection.pq')
out_df


In [ ]:
# outlier_df = pd.DataFrame.from_dict(outlier_dict, orient = 'columns')
# outlier_df.index = deg_ls
# out_df = outlier_df.drop('LSSTerr_null')
# out_df.to_parquet('../paper_data/outlier_rejection.pq')

In [ ]:
mean_stdev_dict = {'TrainZ':[], 
                'CMNN': [], 
                'GPz_GL': [], 
                'FZBoost': [], 
                'PZFlow': []}

In [ ]:
GL_ms_dict = {'GPz_GL': []}

In [ ]:
import numpy as np
import scipy 
from scipy import stats

def MS_OutlierRejection(ls):
    # for i in ls: 
    #     if np.isnan(i)==True:
    #         ls.remove(i)
    mean = np.mean(ls)
    stdev = stats.tstd(ls)
    #print(mean, stdev)
    len_b4 = len(ls)
    for i in ls:
        if np.abs(mean - i) > 3*stdev:
            ls.remove(i) 
    len_aft = len(ls)
    if np.abs(len_b4 - len_aft) > 0:
        return MS_OutlierRejection(ls)
    else:
        return [mean, stdev]

In [ ]:
import pandas as pd

def distanceMS_OutlierRejection(est, deg): 
    df = pd.read_parquet('../paper_data/'+est+'/modes.pq')
    data = []
    z = []
    for i in range(len(df[deg])):
        if np.isnan(df[deg][i])==False:
            data.append(df[deg][i])
            z.append(df['redshift'][i])
    ls = []
    for j in range(len(data)):
        ls.append(np.abs(z[j] - data[j]) / np.sqrt(2))
    return MS_OutlierRejection(ls)

In [ ]:
# for key in mean_stdev_dict:
#     ls = []
#     for deg in deg_ls: 
#         ls.append(distanceMS_OutlierRejection(key, deg))
#     #print(ls)
#     mean_stdev_dict[key] = ls

In [ ]:
for key in mean_stdev_dict:
    ls = []
    for deg in deg_ls: 
        ls.append(distanceMS_OutlierRejection(key, deg))
    #print(ls)
    mean_stdev_dict[key] = ls

In [ ]:
print(deg_ls)

In [ ]:
data = pd.DataFrame.from_dict(mean_stdev_dict, orient = 'columns')
new = data.rename(index = {0: 'BOSS', 1: 'DEEP2', 2: 'GAMA', 3: 'HSC', 4: 'VVDSf02', 5: 'zCOSMOS', 6: 'invz=0.1', 7: 'invz=0.3', 8: 'invz=0.6', 9: 'invz=1.0', 10: 'invz=1.4', 11: 'LSSTerr_only'})

new.to_parquet('../paper_data/outlier_rejection_mean_stdev.pq')


new
# lsdata = data['GPz_GL'].tolist()

In [ ]:
msdf = pd.read_parquet('../paper_data/outlier_rejection_mean_stdev.pq')
#newdf = msdf.drop('LSSTerr_null')
msdf['GPz_GL'] = lsdata
#newdf['GPz_GL'] = lsdata


# Dist-to-point

In [ ]:
# from qp.metrics.base_metric_classes import (DistToPointMetric, DistToDistMetric) 


# pdfs_file = "../paper_data/specSelection_lsstErr_TrainZ/outputs/"+spec+"/output_estimate_TrainZ.hdf5"
# testens = qp.read(pdfs_file)

# DistToPointMetric.CDELossMetric.evaluate(testens[0], 0.5)


## PIT

In [ ]:
deg_ls = ['BOSS', 
          'DEEP2', 
          'GAMA', 
          'HSC', 
          'VVDSf02', 
          'zCOSMOS', 
          'invz=0.1',
          'invz=0.3',
          'invz=0.6',
          'invz=1.0', 
          'invz=1.4', 
          'LSSTerr_only', 
          'LSSTerr_null' 
          #'control'
          ]


In [ ]:
TrainZ_pit_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [],
             'invz=0.3': [],
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [],
             #'control': []
}


In [ ]:
CMNN_pit_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [],
             'invz=0.3': [],
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [],
             #'control': []
}


In [ ]:
GPz_pit_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [],
             'invz=0.3': [],
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             #'control': []
}

GPz_GL_pit_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [],
             'invz=0.3': [],
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             #'control': []
}


deg_ls = ['BOSS', 
          'DEEP2', 
          'GAMA', 
          'HSC', 
          'VVDSf02', 
          'zCOSMOS', 
          'invz=0.1', 
          #'invz=0.2',
          'invz=0.3',
          #'invz=0.4',
          #'invz=0.5',
          'invz=0.6',
          #'invz=0.7',
          #'invz=0.8',
          #'invz=0.9',
          'invz=1.0', 
          #'invz=1.1', 
          #'invz=1.2', 
          #'invz=1.3', 
          'invz=1.4', 
          #'invz=1.5', 
          'LSSTerr_only', 
          'LSSTerr_null', 
          ]

In [ ]:
FZBoost_pit_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [],
             'invz=0.3': [],
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': [], 
             #'control': []
}


In [ ]:
PZFlow_pit_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [],
             'invz=0.3': [],
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': []
}


In [ ]:
def getPIT(est, deg):
    import qp

    ls = []

    if est == 'PZFlow': 
        filepath = "../paper_data/"+est+"/"+deg+"/output_estimate_pzflow.hdf5"
        truth = pd.read_parquet("../paper_data/PZFlow/kld_stats.pq")['redshift']
    elif est =='GPz': 
        filepath = "../paper_data/GPz_GL/"+deg+"/output_estimate_"+est+".hdf5"
        truth = pd.read_parquet("../paper_data/"+est+"/kld_stats.pq")['redshift']
    else:
        filepath = "../paper_data/"+est+"/"+deg+"/output_estimate_"+est+".hdf5"
        truth = pd.read_parquet("../paper_data/"+est+"/kld_stats.pq")['redshift']

    ens = qp.read(filepath)

    for i in range(len(truth)):
        ls.append(ens[i].cdf(truth[i])[0][0])

    print(deg+' done')
    
    return ls


In [ ]:
for deg in deg_ls: 
    GPz_GL_pit_dict[deg] = getPIT('GPz', deg)

# GPz_GL_pit_dict['invz=0.3'] = getPIT('GPz', 'invz=0.3')
# GPz_GL_pit_dict['invz=1.4'] = getPIT('GPz', 'invz=1.4')

In [ ]:
truez = pd.read_parquet("../paper_data/PZFlow/kld_stats.pq")['redshift']

GPz_GL_pit_df = pd.DataFrame.from_dict(GPz_GL_pit_dict, orient = 'columns')
GPz_GL_pit_df.insert(0, 'redshift', truez)
GPz_GL_pit_df.to_parquet('../paper_data/GPz_GL/PIT.pq')

In [ ]:
pd.read_parquet('../paper_data/GPz_GL/PIT.pq')

In [ ]:
for deg in deg_ls: 
    TrainZ_pit_dict[deg] = getPIT('TrainZ', deg)

In [ ]:
truez = pd.read_parquet("../paper_data/PZFlow/kld_stats.pq")['redshift']
# truez2 = pd.read_parquet("../paper_data/PZFlow/control_kld_stats.pq")['redshift']

In [ ]:
for key in TrainZ_pit_dict:
    print(key, len(TrainZ_pit_dict[key]))

In [ ]:
TrainZ_pit_df = pd.DataFrame.from_dict(TrainZ_pit_dict, orient = 'columns')
TrainZ_pit_df.insert(0, 'redshift', truez)
TrainZ_pit_df.to_parquet('../paper_data/TrainZ/PIT.pq')

In [ ]:
pd.read_parquet('../paper_data/TrainZ/PIT.pq')

In [ ]:
for deg in deg_ls: 
    CMNN_pit_dict[deg] = getPIT('CMNN', deg)

In [ ]:
CMNN_pit_df = pd.DataFrame.from_dict(CMNN_pit_dict, orient = 'columns')
CMNN_pit_df.insert(0, 'redshift', truez)
CMNN_pit_df.to_parquet('../paper_data/CMNN/PIT.pq')

In [ ]:
pd.read_parquet('../paper_data/CMNN/PIT.pq')

In [ ]:
for deg in deg_ls: 
    GPz_pit_dict[deg] = getPIT('GPz', deg)

In [ ]:
GPz_pit_df = pd.DataFrame.from_dict(GPz_pit_dict, orient = 'columns')
GPz_pit_df.insert(0, 'redshift', truez)
GPz_pit_df.to_parquet('../paper_data/GPz/PIT.pq')

In [ ]:
pd.read_parquet('../paper_data/GPz/PIT.pq')

In [ ]:
for deg in deg_ls: 
    FZBoost_pit_dict[deg] = getPIT('FZBoost', deg)

In [ ]:
FZBoost_pit_df = pd.DataFrame.from_dict(FZBoost_pit_dict, orient = 'columns')
FZBoost_pit_df.insert(0, 'redshift', truez)
FZBoost_pit_df.to_parquet('../paper_data/FZBoost/PIT.pq')

In [ ]:
pd.read_parquet('../paper_data/FZBoost/PIT.pq')

In [ ]:
for deg in deg_ls: 
    PZFlow_pit_dict[deg] = getPIT('PZFlow', deg)

In [ ]:
for key in PZFlow_pit_dict: 
    print(key, len(PZFlow_pit_dict[key]))

In [ ]:
PZFlow_pit_df = pd.DataFrame.from_dict(PZFlow_pit_dict, orient = 'columns')
PZFlow_pit_df.insert(0, 'redshift', truez)
PZFlow_pit_df.to_parquet('../paper_data/PZFlow/PIT.pq')

In [ ]:
pd.read_parquet('../paper_data/PZFlow/PIT.pq')

In [ ]:
lets make an error 

### old

In [ ]:
pt_stat_ls = [60, 4]

pit_dict_ls = [[TrainZ_pit_dict, 'TrainZ'], [CMNN_pit_dict, 'CMNN'], [GPz_pit_dict, 'GPz'], [FZBoost_pit_dict, 'FZBoost']]

def makePITHists(deg, grid, axes, ylim):
    for dict in pit_dict_ls:
        axes.hist(dict[0][deg], bins = grid, density = True, alpha = 0.25, label = dict[1])
    x = np.linspace(0, 1, 100)
    axes.plot(x, uniform.pdf(x), label = 'Uniform Distribution')
    axes.set_ylim(0, ylim)
    axes.legend()

In [ ]:
from scipy.stats import uniform

fig, axes = plt.subplots(nrows = 2, ncols = 9, figsize = (16*4, 16)) 
ct1 = 0
for pt_stat in pt_stat_ls:
    ct2 = 0
    for deg in deg_ls:
        makePITHists(deg, np.linspace(0, 1, 100), axes[ct1][ct2], pt_stat)
        ct2 += 1
    ct1 += 1

axes[0][0].set_title('BOSS')
axes[0][1].set_title('DEEP2')
axes[0][2].set_title('GAMA')
axes[0][3].set_title('HSC')
axes[0][4].set_title('VVDSf02')
axes[0][5].set_title('zCOSMOS')
axes[0][6].set_title('invz, pivot = 1.0')
axes[0][7].set_title('invz, pivot = 1.4')
axes[0][8].set_title('control')

axes[0][0].set_ylabel('Probability Integral Transform')
axes[1][0].set_ylabel('Zoomed-in PIT')

fig.supxlabel("PIT Value", size = 'medium')
fig.supylabel("Normalized Bin Count", size = 'medium')
fig.suptitle("Probability Integral Transform of Results for Four Estimators", size = 'xx-large')

## CDE Loss

In [ ]:
import scipy.integrate

In [ ]:
TrainZ_cde_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [],
             'invz=0.3': [],
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': []
}
CMNN_cde_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [],
             'invz=0.3': [],
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': []
}

GPz_cde_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [],
             'invz=0.3': [],
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': []
}

GPz_GL_cde_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [],
             'invz=0.3': [],
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': []
}

deg_ls = ['BOSS', 
          'DEEP2', 
          'GAMA', 
          'HSC', 
          'VVDSf02', 
          'zCOSMOS', 
          'invz=0.1', 
          'invz=0.2',
          'invz=0.3',
          'invz=0.4',
          'invz=0.5',
          'invz=0.6',
          'invz=0.7',
          'invz=0.8',
          'invz=0.9',
          'invz=1.0', 
          'invz=1.1', 
          'invz=1.2', 
          'invz=1.3', 
          'invz=1.4', 
          'invz=1.5', 
          'LSSTerr_only', 
          'LSSTerr_null', 
          ]

FZBoost_cde_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [],
             'invz=0.3': [],
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': []
}

PZFlow_cde_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [],
             'invz=0.3': [],
             'invz=0.6': [],
             'invz=1.0': [],
             'invz=1.4': [],
             'LSSTerr_only': [], 
             'LSSTerr_null': []
}

truez = pd.read_parquet("../paper_data/PZFlow/kld_stats.pq")['redshift']

In [ ]:
GPz_GL_cde_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             'invz=0.1': [],
             'invz=0.2': [],
             'invz=0.3': [],
             'invz=0.4': [],
             'invz=0.5': [],
             'invz=0.6': [],
             'invz=0.7': [],
             'invz=0.8': [],
             'invz=0.9': [],
             'invz=1.0': [],
             'invz=1.1': [],
             'invz=1.2': [],
             'invz=1.3': [],
             'invz=1.4': [],
             'invz=1.5': [],
             'LSSTerr_only': [], 
}

In [ ]:
def getCDELoss1(est, deg):
    import scipy.integrate

    if est == 'PZFlow': 
        filepath = "../paper_data/"+est+'/'+deg+'/'+'../paper_data/output_estimate_pzflow.hdf5'
    else:
        filepath = "../paper_data/"+est+'/'+deg+'/'+'output_estimate_'+est+'.hdf5'

    file = h5py.File(filepath)
    ens = qp.read(filepath)

    square_ls = []
    pdf_ls = []

    for i in range(len(truez)):
        grid = np.linspace(0, 3, 301)
        arr = ( file['data']['yvals'][i] )**2
        square_ls.append(scipy.integrate.trapezoid(arr , x=grid, dx=0.3))
    print(deg+" square integration done")

    for i in range(len(truez)):
        pdf_ls.append(ens[i].pdf(truez[i]))
    print(deg+" pdf evaluation done")

    term_1 = np.mean(square_ls)
    term_2 = np.mean(pdf_ls)
    
    #TrainZ_cde_dict[deg] = [term_1 - 2*term_2]
    print(deg+": "+ str(term_1 - 2*term_2))
    
    return term_1 - 2*term_2

In [ ]:
def getCDELoss2(est, deg): 
    if est != 'GPz':    
        filepath = "../paper_data/"+est+'/'+deg+'/'+'output_estimate_'+est+'.hdf5'
    else:
        filepath = "../paper_data/GPz_GL/"+deg+'/'+'output_estimate_GPz.hdf5'

    file = h5py.File(filepath)
    ens = qp.read(filepath)

    square_ls = []
    pdf_ls = []

    for i in range(len(truez)):
        grid = np.linspace(0, 3, 301)
        arr = ( (1. /(file['data']['scale'][i] * np.sqrt(2*np.pi))) * np.exp((-1/2) * ((grid - file['data']['loc'][i])/ file['data']['scale'][i] )**2) )**2
        square_ls.append(scipy.integrate.trapezoid(arr , x=grid, dx=0.1))
    print(deg+" square integration done")

    for i in range(len(truez)):
        pdf_ls.append(ens[i].pdf(truez[i]))
    print(deg+" pdf evaluation done")

    term_1 = np.mean(square_ls)
    term_2 = np.mean(pdf_ls)
    
    #CMNN_cde_dict[deg] = [term_1 - 2*term_2]
    print(deg+": "+ str(term_1 - 2*term_2))

    return term_1 - 2*term_2

In [ ]:
def getCDELoss3(est, deg):
    import scipy.integrate

    filepath = "../paper_data/"+est+'/'+deg+'/'+'output_estimate_'+est+'.hdf5'

    file = h5py.File(filepath)
    ens = qp.read(filepath)

    square_ls = []
    pdf_ls = []

    for i in range(len(truez)):
        grid = np.linspace(0, 3, 100)
        arr = ( file['data']['yvals'][i] )**2
        square_ls.append(scipy.integrate.trapezoid(arr , x=grid, dx=0.3))
    print(deg+" square integration done")

    for i in range(len(truez)):
        pdf_ls.append(ens[i].pdf(truez[i]))
    print(deg+" pdf evaluation done")

    term_1 = np.mean(square_ls)
    term_2 = np.mean(pdf_ls)
    
    #FZBoost_cde_dict[deg] = [term_1 - 2*term_2]
    print(deg+": "+ str(term_1 - 2*term_2))

    return term_1 - 2*term_2


In [ ]:
def getCDELoss4(est, deg):
    import scipy.integrate

    if est == 'PZFlow': 
        filepath = "../paper_data/"+est+'/'+deg+'/'+'../paper_data/output_estimate_pzflow.hdf5'
    else:
        filepath = "../paper_data/"+est+'/'+deg+'/'+'output_estimate_'+est+'.hdf5'

    file = h5py.File(filepath)
    ens = qp.read(filepath)

    square_ls = []
    pdf_ls = []

    for i in range(len(truez)):
        grid = np.linspace(0, 3, 301)
        if np.isnan(file['data']['yvals'][i][0])==True:
            pass
        else:
            arr = ( file['data']['yvals'][i] )**2
            square_ls.append(scipy.integrate.trapezoid(arr , x=grid, dx=0.3))
    print(deg+" square integration done")

    for i in range(len(truez)):
        if np.isnan(file['data']['yvals'][i][0])==True:
            pass
        else:
            pdf_ls.append(ens[i].pdf(truez[i]))
    print(deg+" pdf evaluation done")

    term_1 = np.mean(square_ls)
    term_2 = np.mean(pdf_ls)
    
    #TrainZ_cde_dict[deg] = [term_1 - 2*term_2]
    print(deg+": "+ str(term_1 - 2*term_2))
    
    return term_1 - 2*term_2

In [ ]:
# for key in GPz_GL_invz_cde_dict: 
#     GPz_GL_invz_cde_dict[key] = getCDELoss2('GPz', key)
import qp

for key in GPz_GL_cde_dict:
    GPz_GL_cde_dict[key] = getCDELoss2('GPz', key)

In [ ]:
GL_cde_df = pd.DataFrame.from_dict(GPz_GL_cde_dict, orient = 'index')
GL_cde_df.to_parquet("../paper_data/GPz_GL/invz_CDE_losses.pq")
GL_cde_df = pd.read_parquet("../paper_data/GPz_GL/invz_CDE_losses.pq")
GL_cde_df

In [ ]:
newdf = GL_cde_df.drop('invz=0.2')
newdf1 = newdf.drop('invz=0.4') 
newdf2 = newdf1.drop('invz=0.5')
newdf3 = newdf2.drop('invz=0.7')
newdf4 = newdf3.drop('invz=0.8') 
newdf5 = newdf4.drop('invz=0.9') 
newdf6 = newdf5.drop('invz=1.1') 
newdf7 = newdf6.drop('invz=1.2')  
newdf8 = newdf7.drop('invz=1.3')  
newdf9 = newdf8.drop('invz=1.5') 
newdf9

In [ ]:
table = pd.read_parquet('../paper_data/CDE_loss.pq')
newtable = table.drop('LSSTerr_null')
newtable['GPz_GL'] = newdf9[0]
newtable

In [ ]:
newtable.to_parquet('../paper_data/CDE_loss.pq')

In [ ]:
for key in TrainZ_cde_dict: 
    TrainZ_cde_dict[key] = getCDELoss1('TrainZ', key)

In [ ]:
for key in CMNN_cde_dict: 
    CMNN_cde_dict[key] = getCDELoss2('CMNN', key)

In [ ]:
for key in GPz_cde_dict: 
    GPz_cde_dict[key] = getCDELoss2('GPz', key)

In [ ]:
for key in FZBoost_cde_dict: 
    FZBoost_cde_dict[key] = getCDELoss3('FZBoost', key)

In [ ]:
for key in PZFlow_cde_dict: 
    PZFlow_cde_dict[key] = getCDELoss4('PZFlow', key)

In [ ]:
TrainZ_cde_df = pd.DataFrame.from_dict(TrainZ_cde_dict, orient = 'index')

CMNN_cde_df = pd.DataFrame.from_dict(CMNN_cde_dict, orient = 'index')

GPz_cde_df = pd.DataFrame.from_dict(GPz_cde_dict, orient = 'index')

FZBoost_cde_df = pd.DataFrame.from_dict(FZBoost_cde_dict, orient = 'index')

PZFlowcde_df = pd.DataFrame.from_dict(PZFlow_cde_dict, orient = 'index')

# TrainZ_cde_df.insert(0, 'redshift', truez)

In [ ]:
cde_frame = pd.concat([TrainZ_cde_df, CMNN_cde_df, GPz_cde_df, FZBoost_cde_df, PZFlowcde_df,], axis=1)
cde_frame.columns = ['TrainZ', 'CMNN', 'GPz', 'FZBoost', 'PZFlow']
cde_frame.to_parquet("../paper_data/CDE_loss.pq")

In [ ]:
cde_frame = pd.read_parquet("../paper_data/CDE_loss.pq")
#cde_frame['GPz_GL'] =  pd.DataFrame.from_dict(GPz_GL_cde_dict, orient = 'index')[0]

In [ ]:
cde_frame.to_parquet("../paper_data/CDE_loss.pq")

In [ ]:
cde_frame['GPz_GL']['invz=0.3'] = GL_invz_cde_df[0]['invz=0.3']
cde_frame['GPz_GL']['invz=1.4'] = GL_invz_cde_df[0]['invz=1.4']
cde_frame.to_parquet("../paper_data/CDE_loss.pq")
cde_frame = pd.read_parquet("../paper_data/CDE_loss.pq")
cde_frame

In [ ]:
lets make an error

### TrainZ

In [ ]:
eval_dir_TrainZ = "../paper_data/specSelection_lsstErr_TrainZ/evaluation"
os.chdir(eval_dir_TrainZ)

TrainZ_cde_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             '1.0': [],
             '1.4': [],
             'control': []
}

for spec in spec_ls:

    os.chdir("../paper_data/specSelection_lsstErr_TrainZ/evaluation/"+spec)
    pdfs_file = "../paper_data/specSelection_lsstErr_TrainZ/outputs/"+spec+"/output_estimate_TrainZ.hdf5"

    file = h5py.File(pdfs_file)
    ens = qp.read(pdfs_file)

    square_ls = []
    pdf_ls = []

    for i in range(len(point_z_data)):
        grid = np.linspace(0, 3, 301)
        arr = ( file['data']['yvals'][i] )**2
        square_ls.append(scipy.integrate.trapezoid(arr , x=grid, dx=0.3))
    print(spec+" square integration done")

    for i in range(len(point_z_data)):
        pdf_ls.append(ens[i].pdf(point_z_data[i]))
    print(spec+" pdf evaluation done")

    term_1 = np.mean(square_ls)
    term_2 = np.mean(pdf_ls)
    
    TrainZ_cde_dict[spec] = [term_1 - 2*term_2]
    print(spec+": "+ str(term_1 - 2*term_2))



for pivot in pivot_ls:

    os.chdir("../paper_data/invz_lsstErr_TrainZ/evaluation/"+pivot)
    pdfs_file = "../paper_data/invz_lsstErr_TrainZ/outputs/"+pivot+"/output_estimate_TrainZ.hdf5"

    file = h5py.File(pdfs_file)
    ens = qp.read(pdfs_file)

    square_ls = []
    pdf_ls = []

    for i in range(len(point_z_data)):
        grid = np.linspace(0, 3, 301)
        arr = ( file['data']['yvals'][i] )**2
        square_ls.append(scipy.integrate.trapezoid(arr , x=grid, dx=0.3))
    print(pivot+" square integration done")

    for i in range(len(point_z_data)):
        pdf_ls.append(ens[i].pdf(point_z_data[i]))
    print(pivot+" pdf evaluation done")

    term_1 = np.mean(square_ls)
    term_2 = np.mean(pdf_ls)
    
    TrainZ_cde_dict[pivot] = [term_1 - 2*term_2]
    print(pivot+": "+ str(term_1 - 2*term_2))

### CMNN

In [ ]:
eval_dir_CMNN = "../paper_data/specSelection_lsstErr_CMNN/evaluation"
os.chdir(eval_dir_CMNN)

CMNN_cde_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             '1.0': [],
             '1.4': [],
             'control': []
}

for spec in spec_ls:

    os.chdir("../paper_data/specSelection_lsstErr_CMNN/evaluation/"+spec)
    pdfs_file = "../paper_data/specSelection_lsstErr_CMNN/outputs/"+spec+"/output_estimate_CMNN.hdf5"

    file = h5py.File(pdfs_file)
    ens = qp.read(pdfs_file)

    square_ls = []
    pdf_ls = []

    for i in range(len(point_z_data)):
        grid = np.linspace(0, 3, 301)
        arr = ( (1. /(file['data']['scale'][i] * np.sqrt(2*np.pi))) * np.exp((-1/2) * ((grid - file['data']['loc'][i])/ file['data']['scale'][i] )**2) )**2
        square_ls.append(scipy.integrate.trapezoid(arr , x=grid, dx=0.1))
    print(spec+" square integration done")

    for i in range(len(point_z_data)):
        pdf_ls.append(ens[i].pdf(point_z_data[i]))
    print(spec+" pdf evaluation done")

    term_1 = np.mean(square_ls)
    term_2 = np.mean(pdf_ls)
    
    CMNN_cde_dict[spec] = [term_1 - 2*term_2]
    print(spec+": "+ str(term_1 - 2*term_2))



for pivot in pivot_ls:

    os.chdir("../paper_data/invz_lsstErr_CMNN/evaluation/"+pivot)
    pdfs_file = "../paper_data/invz_lsstErr_CMNN/outputs/"+pivot+"/output_estimate_CMNN.hdf5"

    file = h5py.File(pdfs_file)
    ens = qp.read(pdfs_file)

    square_ls = []
    pdf_ls = []

    for i in range(len(point_z_data)):
        grid = np.linspace(0, 3, 301)
        arr = ( (1. /(file['data']['scale'][i] * np.sqrt(2*np.pi))) * np.exp((-1/2) * ((grid - file['data']['loc'][i])/ file['data']['scale'][i] )**2) )**2
        square_ls.append(scipy.integrate.trapezoid(arr , x=grid, dx=0.1))
    print(pivot+" square integration done")

    for i in range(len(point_z_data)):
        pdf_ls.append(ens[i].pdf(point_z_data[i]))
    print(pivot+" pdf evaluation done")

    term_1 = np.mean(square_ls)
    term_2 = np.mean(pdf_ls)
    
    CMNN_cde_dict[pivot] = [term_1 - 2*term_2]
    print(pivot+": "+ str(term_1 - 2*term_2))
    

### GPz

In [ ]:
eval_dir_GPz = "../paper_data/specSelection_lsstErr_GPz/evaluation"
os.chdir(eval_dir_GPz)

GPz_cde_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             '1.0': [],
             '1.4': [],
             'control': []
}

for spec in spec_ls:

    os.chdir("../paper_data/specSelection_lsstErr_GPz/evaluation/"+spec)
    pdfs_file = "../paper_data/specSelection_lsstErr_GPz/outputs/"+spec+"/output_estimate_GPz.hdf5"

    file = h5py.File(pdfs_file)
    ens = qp.read(pdfs_file)

    square_ls = []
    pdf_ls = []

    for i in range(len(point_z_data)):
        grid = np.linspace(0, 3, 301)
        arr = ( (1. /(file['data']['scale'][i] * np.sqrt(2*np.pi))) * np.exp((-1/2) * ((grid - file['data']['loc'][i])/ file['data']['scale'][i] )**2) )**2
        square_ls.append(scipy.integrate.trapezoid(arr , x=grid, dx=0.1))
    print(spec+" square integration done")

    for i in range(len(point_z_data)):
        pdf_ls.append(ens[i].pdf(point_z_data[i]))
    print(spec+" pdf evaluation done")

    term_1 = np.mean(square_ls)
    term_2 = np.mean(pdf_ls)
    
    GPz_cde_dict[spec] = [term_1 - 2*term_2]
    print(spec+": "+ str(term_1 - 2*term_2))



for pivot in pivot_ls:

    os.chdir("../paper_data/invz_lsstErr_GPz/evaluation/"+pivot)
    pdfs_file = "../paper_data/invz_lsstErr_GPz/outputs/"+pivot+"/output_estimate_GPz.hdf5"

    file = h5py.File(pdfs_file)
    ens = qp.read(pdfs_file)

    square_ls = []
    pdf_ls = []

    for i in range(len(point_z_data)):
        grid = np.linspace(0, 3, 301)
        arr = ( (1. /(file['data']['scale'][i] * np.sqrt(2*np.pi))) * np.exp((-1/2) * ((grid - file['data']['loc'][i])/ file['data']['scale'][i] )**2) )**2
        square_ls.append(scipy.integrate.trapezoid(arr , x=grid, dx=0.1))
    print(pivot+" square integration done")

    for i in range(len(point_z_data)):
        pdf_ls.append(ens[i].pdf(point_z_data[i]))
    print(pivot+" pdf evaluation done")

    term_1 = np.mean(square_ls)
    term_2 = np.mean(pdf_ls)
    
    GPz_cde_dict[pivot] = [term_1 - 2*term_2]
    print(pivot+": "+ str(term_1 - 2*term_2))
    

### FZBoost

In [ ]:
eval_dir_FZBoost = "../paper_data/specSelection_lsstErr_FZBoost/evaluation"
os.chdir(eval_dir_FZBoost)

FZBoost_cde_dict = {'BOSS': [], 
             'DEEP2': [],
             'GAMA': [],
             'HSC': [],
             'VVDSf02': [],
             'zCOSMOS': [],
             '1.0': [],
             '1.4': [],
             'control': []
}

for spec in spec_ls:

    os.chdir("../paper_data/specSelection_lsstErr_FZBoost/evaluation/"+spec)
    pdfs_file = "../paper_data/specSelection_lsstErr_FZBoost/outputs/"+spec+"/output_estimate_FZBoost.hdf5"

    file = h5py.File(pdfs_file)
    ens = qp.read(pdfs_file)

    square_ls = []
    pdf_ls = []

    for i in range(len(point_z_data)):
        grid = np.linspace(0, 3, 100)
        arr = ( file['data']['yvals'][i] )**2
        square_ls.append(scipy.integrate.trapezoid(arr , x=grid, dx=0.3))
    print(spec+" square integration done")

    for i in range(len(point_z_data)):
        pdf_ls.append(ens[i].pdf(point_z_data[i]))
    print(spec+" pdf evaluation done")

    term_1 = np.mean(square_ls)
    term_2 = np.mean(pdf_ls)
    
    FZBoost_cde_dict[spec] = [term_1 - 2*term_2]
    print(spec+": "+ str(term_1 - 2*term_2))



for pivot in pivot_ls:

    os.chdir("../paper_data/invz_lsstErr_FZBoost/evaluation/"+pivot)
    pdfs_file = "../paper_data/invz_lsstErr_FZBoost/outputs/"+pivot+"/output_estimate_FZBoost.hdf5"

    file = h5py.File(pdfs_file)
    ens = qp.read(pdfs_file)

    square_ls = []
    pdf_ls = []

    for i in range(len(point_z_data)):
        grid = np.linspace(0, 3, 100)
        arr = ( file['data']['yvals'][i] )**2
        square_ls.append(scipy.integrate.trapezoid(arr , x=grid, dx=0.3))
    print(pivot+" square integration done")

    for i in range(len(point_z_data)):
        pdf_ls.append(ens[i].pdf(point_z_data[i]))
    print(pivot+" pdf evaluation done")

    term_1 = np.mean(square_ls)
    term_2 = np.mean(pdf_ls)
    
    FZBoost_cde_dict[pivot] = [term_1 - 2*term_2]
    print(pivot+": "+ str(term_1 - 2*term_2))

# Visualizing degradation

In [ ]:
# orig = pd.read_parquet("../paper_data/specSelection_lsstErr_TrainZ/outputs/BOSS/output_train_set.pq")
orig = pd.read_parquet("../paper_data/invz_lsstErr_TrainZ/outputs/control/output_lsst_error_0.pq")
test = pd.read_parquet("../paper_data/specSelection_lsstErr_TrainZ/outputs/BOSS/output_lsst_error.pq")

# BOSS = pd.read_parquet("../paper_data/specSelection_lsstErr_TrainZ/outputs/BOSS/output_specselection_boss.pq")
BOSS = pd.read_parquet("../paper_data/specSelection_lsstErr_TrainZ/outputs/BOSS/output_lsst_error_0.pq")

# DEEP2 = pd.read_parquet("../paper_data/specSelection_lsstErr_TrainZ/outputs/DEEP2/output_specselection_deep2.pq")
DEEP2 = pd.read_parquet("../paper_data/specSelection_lsstErr_TrainZ/outputs/DEEP2/output_lsst_error_0.pq")

# GAMA = pd.read_parquet("../paper_data/specSelection_lsstErr_TrainZ/outputs/GAMA/output_specselection_gama.pq")
GAMA = pd.read_parquet("../paper_data/specSelection_lsstErr_TrainZ/outputs/GAMA/output_lsst_error_0.pq")

# HSC = pd.read_parquet("../paper_data/specSelection_lsstErr_TrainZ/outputs/HSC/output_specselection_HSC.pq")
HSC = pd.read_parquet("../paper_data/specSelection_lsstErr_TrainZ/outputs/HSC/output_lsst_error_0.pq")

# VVDSf02 = pd.read_parquet("../paper_data/specSelection_lsstErr_TrainZ/outputs/VVDSf02/output_specselection_VVDSf02.pq")
VVDSf02 = pd.read_parquet("../paper_data/specSelection_lsstErr_TrainZ/outputs/VVDSf02/output_lsst_error_0.pq")

# zCOSMOS = pd.read_parquet("../paper_data/specSelection_lsstErr_TrainZ/outputs/zCOSMOS/output_specselection_zCOSMOS.pq")
zCOSMOS = pd.read_parquet("../paper_data/specSelection_lsstErr_TrainZ/outputs/zCOSMOS/output_lsst_error_0.pq")

# _10 = pd.read_parquet("../paper_data/invz_lsstErr_TrainZ/outputs/1.0/output_inv_redshift.pq")
_10 = pd.read_parquet("../paper_data/invz_lsstErr_TrainZ/outputs/1.0/output_lsst_error_0.pq")

# _14 = pd.read_parquet("../paper_data/invz_lsstErr_TrainZ/outputs/1.4/output_inv_redshift.pq")
_14 = pd.read_parquet("../paper_data/invz_lsstErr_TrainZ/outputs/1.4/output_lsst_error_0.pq")

# control = pd.read_parquet("../paper_data/invz_lsstErr_TrainZ/outputs/control/output_train_set.pq")
control = pd.read_parquet("../paper_data/invz_lsstErr_TrainZ/outputs/control/output_lsst_error_0.pq")

In [ ]:
degs = [[BOSS, 0], [DEEP2,1], [GAMA,2], [HSC,3], [VVDSf02,4], [zCOSMOS,5], [_10, 6], [_14, 7], [control, 8]]
bands = [['mag_u_lsst',0], ['mag_g_lsst',1], ['mag_r_lsst',2], ['mag_i_lsst',3], ['mag_z_lsst',4], ['mag_y_lsst',5]]

fig, axes = plt.subplots(nrows = len(bands),  ncols = len(degs), figsize = (20, 12)) 

for i in bands:
    for j in degs:
        #axes[i[1]][j[1]].hist2d(orig['redshift'], orig[i[0]], bins = 100)
       # axes[i[1]][j[1]].contour(j[0]['redshift'], j[0][i[0]], s=0.5, alpha=0.25, color = 'orange')
        axes[i[1]][j[1]].scatter(orig['redshift'], orig[i[0]], s=0.5, alpha = 0.25)
        axes[i[1]][j[1]].scatter(j[0]['redshift'], j[0][i[0]], s=0.5, alpha=0.05, color = 'orange')
        #axes[i[1]][j[1]].scatter(test['redshift'], test[i[0]], s=0.5, color = 'lightgreen', alpha = 0.1)
        
axes[0][0].set_ylabel("u band")
axes[1][0].set_ylabel("g band")
axes[2][0].set_ylabel("r band")
axes[3][0].set_ylabel("i band")
axes[4][0].set_ylabel("z band")
axes[5][0].set_ylabel("y band")

axes[0][0].set_title("BOSS")
axes[0][1].set_title("DEEP2")
axes[0][2].set_title("GAMA")
axes[0][3].set_title("HSC")
axes[0][4].set_title("VVDSf02")
axes[0][5].set_title("zCOSMOS")
axes[0][6].set_title("1.0")
axes[0][7].set_title("1.4")
axes[0][8].set_title("control")

fig.supxlabel("Redshift")
fig.supylabel("Magnitude")
fig.suptitle("Training set degradation with spectroscopic surveys and inverse redshift incompleteness")

print("Magnitude vs. redshift for spectroscopic survey degraders")

In [ ]:
help(plt.contour)

In [ ]:
import h5py

spec_ls = ['BOSS',
           'DEEP2', 
           'GAMA', 
           'HSC', 
           'VVDSf02', 
           'zCOSMOS', 
           'LSSTerr_only'
           ]

pzflow_success_dict = {'BOSS': 0, 
                    'DEEP2': 0, 
                    'GAMA': 0,
                    'HSC': 0, 
                    'VVDSf02': 0, 
                    'zCOSMOS': 0,
                    'LSSTerr_only': 0
                    }

for spec in spec_ls: 
    file = h5py.File('../paper_data/PZFlow/'+spec+'/output_estimate_pzflow.hdf5')
    ct = 0
    for i in range(0, 31247):
        if np.isnan(file['data']['yvals'][i][0]) == False:
            ct += 1
    print(spec + ' done')
    pzflow_success_dict[spec] = ct

#print(np.isnan(file['data']['yvals'][1075]))

In [ ]:
pzs_df = pd.DataFrame.from_dict(pzflow_success_dict, orient='index')

pzs_df.to_parquet('../paper_data/PZFlow/success_rates.pq')

In [ ]:
((31220 - pzs_df[0]['VVDSf02']) / 31220 ) * 100